# ESP Failure Prediction — Samsung Innovation Campus CapstoneAI/ML pipeline for predicting Electrical Submersible Pump (ESP) failures and selecting a safe operating frequency.**Everything runs here — no data to download.** The SCADA data is simulated from scratch by Step 3.## What this builds**16 wells x 6 months x 1-minute SCADA data (4.19M rows)**, with 6 mixed failure causes:`GAS_LOCK`, `UNDERLOAD`, `HIGH_TEMP`, `HIGH_DISCHARGE`, `VIBRATION`, `LOW_VOLTAGE`.Wells 1-7 and 11-16 are for training; **wells 8, 9, 10 are held out entirely** and the model never sees them.Two independent branches (they do **not** feed each other):- **Unsupervised** — K-Means + DBSCAN find operating regimes and anomalies, with no failure labels.- **Supervised, two stages** — Stage 1 predicts *will this well trip in the next 30 minutes?*; Stage 2 predicts *which failure cause?*Each well also has a real ESP equipment configuration (Motor / Protector / Gas Separator / Pump), which defines its **safe operating Hz band** — any frequency recommendation is clipped to that well's own equipment limits.## RuntimeAbout **10-12 minutes total** on a free Colab CPU runtime (Step 3 ~5 min, Step 4 ~6 min). No GPU needed.Run the cells in order, top to bottom.

## Step 0 — Install dependencies

### Install the required librariesThis installs every library the whole notebook needs (not just `xgboost` like the original Colab version), because a local VS Code environment does not come with them pre-installed the way Colab does.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib xgboost ipykernel


### Create the working folderAll output files (CSVs and images) will be saved in this local `esp_multiwell` folder next to the notebook.

In [ ]:
import os
os.makedirs('./esp_multiwell', exist_ok=True)
print('ready')

## Step 1 — Well rosterThis is the **only file you edit to add a well.** Add a row (`well_id`, cause profile, `train`/`test`) and re-run from Step 2 — everything else picks it up automatically. A well marked `test` is simulated but never trained on, which is how you test the model against new data.

### Define the well roster (16 wells)A CSV text block that defines, for each well: its ID, its likely failure cause(s) (`name:weight`), and whether it's a train or test well. This is the **single source of truth** that every later step relies on.

In [ ]:
wells_config = '''well_id,causes,split1,GAS_LOCK:1.0,train2,UNDERLOAD:1.0,train3,HIGH_TEMP:1.0,train4,HIGH_DISCHARGE:1.0,train5,VIBRATION:1.0,train6,LOW_VOLTAGE:1.0,train7,GAS_LOCK:0.5;HIGH_TEMP:0.5,train8,UNDERLOAD:0.4;VIBRATION:0.6,test9,HIGH_DISCHARGE:0.5;LOW_VOLTAGE:0.5,test10,GAS_LOCK:0.3;VIBRATION:0.3;LOW_VOLTAGE:0.4,test11,UNDERLOAD:1.0,train12,HIGH_TEMP:1.0,train13,HIGH_DISCHARGE:1.0,train14,VIBRATION:1.0,train15,LOW_VOLTAGE:1.0,train16,GAS_LOCK:1.0,train'''

### Write the roster to a CSV file on disk

In [ ]:
with open('./esp_multiwell/wells_config.csv', 'w') as f:    f.write(wells_config)

### Read the saved file back and display it as a table, to confirm it's correct

In [ ]:
import pandas as pdprint(pd.read_csv('./esp_multiwell/wells_config.csv').to_string(index=False))

## Step 2 — ESP equipment specs and safe Hz bandDerives each well's safe operating frequency band as the intersection of its Motor, Protector and Pump ratings. Wells prone to gas lock are deliberately given a weaker (or no) gas separator, so the equipment explains the failure mode rather than being arbitrary.

### The general ideaFor each well we work out a **safe operating frequency band** (Hz), as the overlap of three real equipment limits: the Motor, the Protector, and the Pump. Wells prone to `GAS_LOCK` are deliberately given a weaker gas separator, so that failure cause is tied to real equipment rather than being an arbitrary label.`Well Safe Hz Band = [largest of all component minimums, smallest of all component maximums]`

In [ ]:
import numpy as npimport pandas as pdrng = np.random.default_rng(77)

### Motor catalogEach row: (model, rated horsepower, rated voltage, rated current, min Hz, max Hz).

In [ ]:
MOTOR_CATALOG = [    # (model, rated_hp, rated_v, rated_a, min_hz, max_hz)    ("HS 456 Series 562 Motor", 456, 2450, 68, 35, 70),    ("HS 562 Series 562 Motor", 562, 2790, 72, 35, 75),    ("HD 375 Series 456 Motor", 375, 2100, 61, 30, 68),    ("HT 500 Series 562 Motor", 500, 2600, 70, 35, 72),]

### Protector catalogEach row: (model, thrust rating in lb, max Hz).

In [ ]:
PROTECTOR_CATALOG = [    # (model, thrust_rating_lb, max_hz)    ("Tandem Protector - Labyrinth/Bag, 2500 lb", 2500, 68),    ("Tandem Protector - Positive Seal, 3200 lb", 3200, 72),    ("Single Protector - Labyrinth, 1800 lb", 1800, 65),]

### Gas separator catalogEach row: (model, type, efficiency %).

In [ ]:
GAS_SEPARATOR_CATALOG = [    # (model, type, efficiency_pct)    ("Rotary Gas Separator, RGS-538", "Rotary", 85),    ("Reverse-Flow Gas Separator, RFS-400", "Reverse-Flow", 65),    ("None (natural separation only)", "None", 25),]

### Pump catalogEach row: (model, stage type, number of stages, min Hz, max Hz, best-efficiency-point Hz).

In [ ]:
PUMP_CATALOG = [    # (model, stage_type, n_stages, min_hz, max_hz, bep_hz)    ("Mixed-Flow Pump, GC-2700, 92 stages", "Mixed-Flow", 92, 40, 65, 52),    ("Mixed-Flow Pump, GC-3300, 78 stages", "Mixed-Flow", 78, 38, 67, 54),    ("Radial-Flow Pump, D-1750, 130 stages", "Radial-Flow", 130, 35, 63, 50),    ("Mixed-Flow Pump, GC-4000, 65 stages", "Mixed-Flow", 65, 40, 70, 55),]

### Load each well's failure cause(s) from `wells_config.csv`

In [ ]:
def load_wells_config(path="./esp_multiwell/wells_config.csv"):    cfg = pd.read_csv(path)    causes_by_well = {}    for _, row in cfg.iterrows():        pairs = str(row["causes"]).split(";")        causes_by_well[int(row["well_id"])] = [p.split(":")[0] for p in pairs]    return causes_by_wellWELL_CAUSES = load_wells_config()

### Build the equipment spec for every wellFor each well: randomly pick a motor, a protector, and a pump, giving a weaker gas separator to wells prone to `GAS_LOCK`, then compute the safe frequency band (`safe_min_hz`, `safe_max_hz`).

In [ ]:
rows = []for well_id, causes in WELL_CAUSES.items():    motor = MOTOR_CATALOG[rng.integers(len(MOTOR_CATALOG))]    protector = PROTECTOR_CATALOG[rng.integers(len(PROTECTOR_CATALOG))]    pump = PUMP_CATALOG[rng.integers(len(PUMP_CATALOG))]    # Wells with GAS_LOCK in their cause profile are deliberately given a    # weaker gas separator (or none) - this is WHY they're gas-lock-prone,    # tying the equipment config to the failure mode rather than being arbitrary    if "GAS_LOCK" in causes:        gas_sep = GAS_SEPARATOR_CATALOG[rng.choice([1, 2])]  # Reverse-Flow or None    else:        gas_sep = GAS_SEPARATOR_CATALOG[rng.choice([0, 1], p=[0.7, 0.3])]  # mostly good Rotary    min_hz = max(motor[4], pump[3])    max_hz = min(motor[5], protector[2], pump[4])    rows.append({        "well_id": well_id,        "motor_model": motor[0], "motor_rated_hp": motor[1], "motor_rated_v": motor[2], "motor_rated_a": motor[3],        "motor_min_hz": motor[4], "motor_max_hz": motor[5],        "protector_model": protector[0], "protector_thrust_rating_lb": protector[1], "protector_max_hz": protector[2],        "gas_separator_model": gas_sep[0], "gas_separator_type": gas_sep[1], "gas_separator_efficiency_pct": gas_sep[2],        "pump_model": pump[0], "pump_stage_type": pump[1], "pump_n_stages": pump[2],        "pump_min_hz": pump[3], "pump_max_hz": pump[4], "pump_bep_hz": pump[5],        "safe_min_hz": min_hz, "safe_max_hz": max_hz,        "dominant_causes": ", ".join(causes),    })

### Collect the results into a table and sanity-check themWe confirm every well has a valid band (min below max) and that its best-efficiency point actually falls inside that band.

In [ ]:
specs = pd.DataFrame(rows).sort_values("well_id").reset_index(drop=True)# sanity check: every well needs a valid non-empty band, and BEP should fall inside itassert (specs["safe_min_hz"] < specs["safe_max_hz"]).all(), "invalid band for some well"specs["bep_in_band"] = specs.apply(lambda r: r["pump_bep_hz"] >= r["safe_min_hz"] and r["pump_bep_hz"] <= r["safe_max_hz"], axis=1)print(specs[["well_id", "safe_min_hz", "safe_max_hz", "pump_bep_hz", "bep_in_band", "gas_separator_type", "gas_separator_efficiency_pct"]].to_string(index=False))

### Save the equipment specs to a CSV file

In [ ]:
specs.to_csv("./esp_multiwell/well_equipment_specs.csv", index=False)print(f"\nSaved well_equipment_specs.csv ({len(specs)} wells)")

## Step 3 — Simulate the SCADA data  ⏱ ~5 minGenerates 1-minute data per well via a state machine (`normal` → `onset` → `shutdown` → `ramp`), with physics-inspired sensor relationships.Note the **precursor leak**: gas accumulation, wear, heat and fouling leak gradually into the sensors *before* a fault officially begins, so detection has genuine early-warning signal to find. `UNDERLOAD` and `LOW_VOLTAGE` deliberately get no precursor — a grid sag is a sudden external event, and that realism shows up later in the per-cause results.This cell is **incremental**: it only simulates wells that aren't already in the CSV, so adding one well later is fast.

### What this step doesIt generates one-minute-resolution SCADA data for every well, using a **state machine** with four states: `normal` -> `onset` -> `shutdown` -> `ramp`. Each of the six failure causes (`GAS_LOCK`, `UNDERLOAD`, `HIGH_TEMP`, `HIGH_DISCHARGE`, `VIBRATION`, `LOW_VOLTAGE`) leaves a different fingerprint on the sensors. There is also a **"precursor leak"**: a small part of the underlying degradation shows up in the sensors even before `onset` officially starts, so the model later has a real early-warning signal to learn from. Generation is **incremental**: a well that's already in the output file is never re-simulated.

### General setup: imports, random seed, simulation length, and file paths

In [ ]:
import osimport numpy as npimport pandas as pdrng = np.random.default_rng(2026)MINUTES_PER_DAY = 24 * 60N_DAYS = 182N = N_DAYS * MINUTES_PER_DAYCAUSES = ["GAS_LOCK", "UNDERLOAD", "HIGH_TEMP", "HIGH_DISCHARGE", "VIBRATION", "LOW_VOLTAGE"]FREQ_RELATED = {"GAS_LOCK": True, "UNDERLOAD": False, "HIGH_TEMP": True,                 "HIGH_DISCHARGE": True, "VIBRATION": False, "LOW_VOLTAGE": False}OUT_DIR = "./esp_multiwell"OUT_PATH = f"{OUT_DIR}/multiwell_scada_10wells_6mo.csv"CONFIG_PATH = f"{OUT_DIR}/wells_config.csv"

### Load each well's profile (failure causes and their weights)

In [ ]:
def load_well_profiles(path=CONFIG_PATH):    cfg = pd.read_csv(path)    profiles = {}    for _, row in cfg.iterrows():        pairs = str(row["causes"]).split(";")        causes = {}        for p in pairs:            name, weight = p.split(":")            causes[name] = float(weight)        profiles[int(row["well_id"])] = {"causes": causes, "split": row["split"]}    return profilesWELL_PROFILES = load_well_profiles()

### Physics constants for the simulation, and loading the equipment specs (from Step 2)

In [ ]:
P_RES = 2600.0K_DRAWDOWN = 9.0K_BOOST = 13.5# Loaded from well_equipment_specs.csv (run equipment_specs.py first) - each# well's operating point and reduced-frequency restart point are now derived# from ITS OWN equipment-rated safe band, not a shared global constant._specs = pd.read_csv(f"{OUT_DIR}/well_equipment_specs.csv").set_index("well_id")

### The `simulate_well` functionThis function is the heart of the simulation. It stays in one cell because it's one tightly-coupled block of logic (internal state that changes minute by minute), but its inner structure breaks down clearly into:1. **Setup** - read this well's own safe frequency band, and initialize the degradation indexes (`gas_accum`, `wear_index`, `thermal_load`, `fouling_index`) with small random starting values.2. **The state machine** - every minute (the `for i in range(N)` loop):   - **normal**: the degradation indexes grow slowly; when one crosses its threshold (1.0), `onset` begins.   - **onset**: the chance of shutdown climbs every minute until `shutdown` actually happens.   - **shutdown**: frequency drops to zero for a random duration, then the index that caused the trip is reset.   - **ramp**: a gradual restart at reduced frequency, ramping back up to normal.3. **The precursor leak** - a small fraction of each degradation index bleeds into its related sensor even during the normal state, growing as that index approaches its trigger threshold.4. **Sensor synthesis** - each active failure cause has a different effect (for example, `GAS_LOCK` lowers current, `LOW_VOLTAGE` drops voltage suddenly). Every random value here now uses `local_rng` (this well's own seed) - this is exactly the consistency fix made in the earlier review.

In [ ]:
def simulate_well(well_id, causes_profile, seed):    local_rng = np.random.default_rng(seed)    rows = []    shutdown_events = []    spec = _specs.loc[well_id]    SAFE_MIN_HZ = float(spec["safe_min_hz"])    SAFE_MAX_HZ = float(spec["safe_max_hz"])    FREQ_BASE = float(spec["pump_bep_hz"])              # normal operating point = pump's best-efficiency Hz    FREQ_RESTART = SAFE_MIN_HZ + 2.0                     # reduced-freq restart, just above the equipment floor    gas_sep_efficiency = float(spec["gas_separator_efficiency_pct"])    # weaker gas separator -> faster gas accumulation for wells prone to GAS_LOCK    gas_lock_multiplier = (100.0 - gas_sep_efficiency) / 75.0    reservoir_pressure = local_rng.uniform(1800, 3200)    productivity_index = local_rng.uniform(0.8, 2.5)    baseline_supply_voltage = local_rng.uniform(4100, 4200)    state = "normal"    freq = FREQ_BASE    gas_accum = local_rng.uniform(0.05, 0.15)    wear_index = local_rng.uniform(0.05, 0.15)      # mechanical wear (vibration cause)    thermal_load = local_rng.uniform(0.05, 0.15)    # cumulative thermal stress (high temp cause)    fouling_index = local_rng.uniform(0.05, 0.15)   # discharge restriction buildup    onset_minutes = 0    shutdown_minutes = 0    shutdown_target = None    ramp_minutes = 0    ramp_duration = None    current_cause = None    Tm, Ti = 178.0, 182.0    def days_to_next_onset():        return local_rng.uniform(3.0, 9.0) * MINUTES_PER_DAY    accel_rate = 1.0 / days_to_next_onset()    for i in range(N):        cause = current_cause        # ---------------- state machine ----------------        if state == "normal":            freq = FREQ_BASE            gas_accum += accel_rate * gas_lock_multiplier * (freq / FREQ_BASE) ** 2 * local_rng.normal(1.0, 0.1) * 0.5            wear_index += accel_rate * local_rng.normal(1.0, 0.15) * 0.35            thermal_load += accel_rate * local_rng.normal(1.0, 0.15) * 0.35            fouling_index += accel_rate * local_rng.normal(1.0, 0.15) * 0.35            gas_accum, wear_index = max(gas_accum, 0), max(wear_index, 0)            thermal_load, fouling_index = max(thermal_load, 0), max(fouling_index, 0)            voltage_sag = local_rng.random() < 0.0001  # rare abrupt supply sag event            triggers = {                "GAS_LOCK": gas_accum >= 1.0,                "UNDERLOAD": local_rng.random() < 0.00008,  # sporadic inflow starvation                "HIGH_TEMP": thermal_load >= 1.0,                "HIGH_DISCHARGE": fouling_index >= 1.0,                "VIBRATION": wear_index >= 1.0,                "LOW_VOLTAGE": voltage_sag,            }            active = [c for c in CAUSES if triggers[c] and c in causes_profile]            if active:                weights = np.array([causes_profile[c] for c in active])                current_cause = local_rng.choice(active, p=weights / weights.sum())                state = "onset"                onset_minutes = 0        elif state == "onset":            freq = FREQ_BASE            onset_minutes += 1            p_trip = min(0.02 * onset_minutes, 0.9)            if local_rng.random() < p_trip:                state = "shutdown"                shutdown_minutes = 0                shutdown_target = local_rng.uniform(90, 240)                shutdown_events.append((i, current_cause))        elif state == "shutdown":            freq = 0.0            shutdown_minutes += 1            if shutdown_minutes >= shutdown_target:                state = "ramp"                ramp_minutes = 0                ramp_duration = local_rng.uniform(300, 600)                freq = FREQ_RESTART                # reset the specific stressor that caused this trip                if current_cause == "GAS_LOCK":                    gas_accum = local_rng.uniform(0.05, 0.15)                elif current_cause == "VIBRATION":                    wear_index = local_rng.uniform(0.05, 0.15)                elif current_cause == "HIGH_TEMP":                    thermal_load = local_rng.uniform(0.05, 0.15)                elif current_cause == "HIGH_DISCHARGE":                    fouling_index = local_rng.uniform(0.05, 0.15)                accel_rate = 1.0 / days_to_next_onset()        elif state == "ramp":            ramp_minutes += 1            frac = min(ramp_minutes / ramp_duration, 1.0)            freq = FREQ_RESTART + frac * (FREQ_BASE - FREQ_RESTART)            if ramp_minutes >= ramp_duration:                state = "normal"                current_cause = None        onset_severity = min(onset_minutes / 40.0, 1.0) if state == "onset" else 0.0        c = cause if state in ("onset", "shutdown") else None        # ---------------- precursor leak (pre-onset, gradual causes only) ----------------        # A small, physically-motivated fraction of each latent index leaks        # into its sensor even during "normal" state, growing as the index        # approaches its trigger threshold (>=1.0). Gives detection a real        # early-warning signal instead of a flat baseline until the instant        # onset officially begins. UNDERLOAD/LOW_VOLTAGE are sporadic/        # external triggers with no accumulator - intentionally no precursor.        precursor_current_dip = 0.05 * min(gas_accum, 1.3)              # GAS_LOCK precursor        precursor_vibration = 0.10 * min(wear_index, 1.3)                # VIBRATION precursor        precursor_tm = 3.0 * min(thermal_load, 1.3)                      # HIGH_TEMP precursor        precursor_pd = 12.0 * min(fouling_index, 1.3)                    # HIGH_DISCHARGE precursor        # ---------------- sensor synthesis ----------------        if state == "shutdown":            current = max(local_rng.normal(0.2, 0.1), 0)            voltage = max(local_rng.normal(30, 15), 0)            vibration = max(local_rng.normal(0.02, 0.01), 0)            pi = P_RES - 5 * (1 - min(shutdown_minutes / shutdown_target, 1.0)) + local_rng.normal(0, 8)            pd_ = max(local_rng.normal(40, 10), 0)            tm_target = 90 + 90 * (1 - min(shutdown_minutes / 200, 1.0))            ti_target = 184 + 2 * min(shutdown_minutes / 200, 1.0)        else:            base_current = 29 + 0.52 * freq            base_voltage = baseline_supply_voltage            vib_base = 0.14 + 0.004 * (freq - 40) + 0.5 * wear_index if wear_index > 1 else 0.14 + 0.004 * (freq - 40)            pi_penalty = 0.0            pd_extra = 0.0            efficiency = 1.0            volt_drop = 0.0            current_mult = 1.0            if c == "GAS_LOCK":                dip = onset_severity * local_rng.uniform(0.15, 0.45)                current_mult *= (1 - dip)                vibration_extra = onset_severity * local_rng.uniform(0.6, 1.8)                pi_penalty = (max(gas_accum - 0.6, 0) ** 1.5) * 55 + onset_severity * local_rng.uniform(0, 90)                efficiency = 1 - 0.55 * onset_severity * local_rng.uniform(0.7, 1.0)            elif c == "UNDERLOAD":                dip = onset_severity * local_rng.uniform(0.25, 0.55)                current_mult *= (1 - dip)                pi_penalty = onset_severity * local_rng.uniform(40, 120)                efficiency = 1 - 0.3 * onset_severity                vibration_extra = onset_severity * local_rng.uniform(0.05, 0.2)            elif c == "HIGH_TEMP":                vibration_extra = onset_severity * local_rng.uniform(0.1, 0.3)                current_mult *= (1 + 0.05 * onset_severity)                efficiency = 1 - 0.1 * onset_severity            elif c == "HIGH_DISCHARGE":                pd_extra = onset_severity * local_rng.uniform(150, 400) + fouling_index * 30 if fouling_index > 1 else onset_severity * local_rng.uniform(150, 400)                current_mult *= (1 + 0.15 * onset_severity)                vibration_extra = onset_severity * local_rng.uniform(0.1, 0.3)                efficiency = 1 - 0.2 * onset_severity            elif c == "VIBRATION":                vibration_extra = onset_severity * local_rng.uniform(1.0, 2.5) + max(wear_index - 1, 0) * 0.8                current_mult *= (1 + 0.03 * onset_severity)                efficiency = 1 - 0.1 * onset_severity            elif c == "LOW_VOLTAGE":                volt_drop = onset_severity * local_rng.uniform(300, 900)                current_mult *= (1 + 0.35 * onset_severity)  # current climbs to compensate                vibration_extra = onset_severity * local_rng.uniform(0.05, 0.15)                efficiency = 1 - 0.15 * onset_severity            else:                vibration_extra = 0.0            # precursor leak applies whenever the well is actually running            # (normal/onset/ramp), on top of any onset-specific effect above            current_mult *= (1 - precursor_current_dip)            vibration_extra += precursor_vibration            pd_extra += precursor_pd            current = base_current * current_mult + local_rng.normal(0, 0.9)            voltage = base_voltage - volt_drop + local_rng.normal(0, 12)            vibration = max(vib_base + vibration_extra + local_rng.normal(0, 0.05), 0)            pi = P_RES - K_DRAWDOWN * freq - pi_penalty + local_rng.normal(0, 12)            pd_ = pi + K_BOOST * freq * efficiency + pd_extra + local_rng.normal(0, 15)            tm_target = (165 + 0.85 * freq + onset_severity * (25 if c == "HIGH_TEMP" else 10)                         + max(thermal_load - 1, 0) * 15 + precursor_tm)            ti_target = 182 + onset_severity * 3 + local_rng.normal(0, 0.3)        Tm += (tm_target - Tm) * 0.02 + local_rng.normal(0, 0.15)        Ti += (ti_target - Ti) * 0.05 + local_rng.normal(0, 0.1)        rows.append((            well_id, i, state, c if c else "", round(freq, 2), round(current, 2), round(voltage, 1),            round(Ti, 2), round(Tm, 2), round(pi, 1), round(pd_, 1), round(vibration, 3),        ))    df = pd.DataFrame(rows, columns=[        "well_id", "minute", "state", "onset_cause", "frequency_hz", "motor_current_a", "voltage_v",        "Ti_intake_temp_f", "Tm_motor_temp_f", "Pi_intake_psi", "Pd_discharge_psi", "vibration_g",    ])    df["shutdown_event"] = 0    df["failure_cause"] = ""    for idx, cause in shutdown_events:        df.loc[idx, "shutdown_event"] = 1        df.loc[idx, "failure_cause"] = cause    df["safe_min_hz"] = SAFE_MIN_HZ    df["safe_max_hz"] = SAFE_MAX_HZ    return df, shutdown_events

### Incremental generation: simulate only the new wells, and append them to the fileIt first checks which wells already exist in `multiwell_scada_10wells_6mo.csv` and skips them, then simulates only the new wells, and finally checks that the generated frequency never violated each well's own safe band.

In [ ]:
existing_well_ids = set()existing_df = Noneif os.path.exists(OUT_PATH):    existing_df = pd.read_csv(OUT_PATH, usecols=["well_id"])    existing_well_ids = set(existing_df["well_id"].unique().tolist())    del existing_dfwells_to_simulate = {wid: p for wid, p in WELL_PROFILES.items() if wid not in existing_well_ids}if not wells_to_simulate:    print("No new wells to simulate - wells_config.csv already fully generated in", OUT_PATH)else:    print(f"Simulating {len(wells_to_simulate)} well(s) not yet in the dataset: {sorted(wells_to_simulate)}")    new_dfs = []    summary = []    for well_id, profile in wells_to_simulate.items():        df_well, events = simulate_well(well_id, profile["causes"], seed=1000 + well_id)        df_well["split"] = profile["split"]        start = pd.Timestamp("2026-01-01 00:00:00")        df_well["timestamp"] = start + pd.to_timedelta(df_well["minute"], unit="m")        new_dfs.append(df_well)        cause_counts = pd.Series([c for _, c in events]).value_counts().to_dict()        summary.append({"well_id": well_id, "split": profile["split"], "n_trips": len(events), **cause_counts})        print(f"well {well_id:2d} [{profile['split']}] causes={list(profile['causes'].keys())}: {len(events)} trips -> {cause_counts}")    new_full = pd.concat(new_dfs, ignore_index=True)    new_full = new_full[[        "well_id", "split", "timestamp", "state", "frequency_hz", "motor_current_a", "voltage_v",        "Ti_intake_temp_f", "Tm_motor_temp_f", "Pi_intake_psi", "Pd_discharge_psi", "vibration_g",        "shutdown_event", "failure_cause", "onset_cause", "safe_min_hz", "safe_max_hz",    ]]    # QA: confirm no simulated frequency ever violated its own well's safe band    violations = new_full[(new_full["frequency_hz"] > 0) &                           ((new_full["frequency_hz"] < new_full["safe_min_hz"] - 0.01) |                            (new_full["frequency_hz"] > new_full["safe_max_hz"] + 0.01))]    print(f"\nSafe-band QA (new wells): {len(violations)} rows violate the well's equipment-rated Hz band "          f"(0 expected; freq=0 rows during shutdown are excluded from this check).")    if os.path.exists(OUT_PATH):        new_full.to_csv(OUT_PATH, mode="a", header=False, index=False)        print(f"\nAppended {len(new_full):,} rows to existing:", OUT_PATH)    else:        new_full.to_csv(OUT_PATH, index=False)        print(f"\nSaved {len(new_full):,} rows to new file:", OUT_PATH)    print("\nSummary (new wells only):")    print(pd.DataFrame(summary).fillna(0))

### Overall summary: total wells, total rows, and shutdown count per well

In [ ]:
full = pd.read_csv(OUT_PATH, usecols=["well_id", "split", "shutdown_event"])print("\nFull dataset now covers", full["well_id"].nunique(), "wells,", len(full), "rows")print(full.groupby(["well_id", "split"])["shutdown_event"].sum().rename("n_trips"))

## Step 4 — Full analysis  ⏱ ~6 minRuns the unsupervised branch, both supervised stages, all baselines, the alarm-policy tuning, and produces every chart.Key design decisions in here, all measured rather than assumed:- **Leakage prevention** — split by well *and* by time (final 6 weeks of training wells held out as validation). Model selection uses validation only, never test.- **Per-well alert thresholds** — a single global probability cutoff does not transfer between wells; each well gets its cutoff from its own score distribution.- **Alarm persistence** — an alarm needs 5 consecutive minutes above threshold, and nuisance alarms are counted as *events*, not minutes.- **Cost-weighted tuning** — `MISS_COST_RATIO` (currently 200:1) is the one knob that moves the operating point along the catch-rate / false-alarm curve.

### Overview of this stepThis is the biggest step in the project, with two independent branches:- **Unsupervised branch**: K-Means and DBSCAN to cluster operating states - fully independent, it does not feed into the supervised branch.- **Supervised branch**, with two stages: **Stage 1** is binary detection (will a shutdown happen within H minutes?), and **Stage 2** classifies the failure cause once one is detected.The key design choice: every feature is z-score normalized **per well** (because each well has a different baseline), with strict leakage prevention - the split is done both by well (wells 8, 9, 10 are never seen during training) and by time (the last 6 weeks of the training wells are used only for validation).

In [ ]:
import warningswarnings.filterwarnings("ignore")import gcimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use("Agg")import matplotlib.pyplot as pltfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.cluster import KMeans, DBSCANfrom sklearn.metrics import silhouette_scorefrom sklearn.decomposition import PCAfrom sklearn.preprocessing import LabelEncoderfrom sklearn.metrics import (    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,    confusion_matrix, classification_report,)from xgboost import XGBClassifierOUT = "./esp_multiwell"H = 30  # failure horizon in minutes - shorter horizon + run-life feature combined (best result)CAUSE_NAMES = ["GAS_LOCK", "UNDERLOAD", "HIGH_TEMP", "HIGH_DISCHARGE", "VIBRATION", "LOW_VOLTAGE"]

### Load the SCADA data and equipment specs, and identify train/test wells

In [ ]:
df = pd.read_csv(f"{OUT}/multiwell_scada_10wells_6mo.csv", parse_dates=["timestamp"])df = df.sort_values(["well_id", "timestamp"]).reset_index(drop=True)df = df.drop(columns=["Ti_intake_temp_f"])  # not used as a modeling feature - drop early to save memoryspecs = pd.read_csv(f"{OUT}/well_equipment_specs.csv").set_index("well_id")TRAIN_WELLS = sorted(df.loc[df["split"] == "train", "well_id"].unique())TEST_WELLS = sorted(df.loc[df["split"] == "test", "well_id"].unique())print("Train wells (fully seen):", TRAIN_WELLS)print("Test wells (fully held out):", TEST_WELLS)SENSOR_COLS = ["motor_current_a", "voltage_v", "vibration_g", "Pi_intake_psi", "Pd_discharge_psi", "Tm_motor_temp_f"]for col in SENSOR_COLS + ["frequency_hz"]:    df[col] = df[col].astype("float32")

### Rolling featuresFor each sensor: mean/standard deviation over 15, 60 and 240-minute windows, plus the change over the last 5 minutes. Computed with `groupby().transform()` directly on the main table instead of copying each well separately, to save memory with 16 wells.

In [ ]:
# ============================================================RAW_FEATURES = ["frequency_hz", "minutes_since_last_shutdown"] + SENSOR_COLSROLL_STATS = ["roll15_mean", "roll15_std", "roll60_mean", "roll60_std", "roll240_mean", "chg5"]ROLL_FEATURES = [f"{c}_{stat}" for c in SENSOR_COLS for stat in ROLL_STATS]ALL_FEATURES = RAW_FEATURES + ROLL_FEATURESgrouped = df.groupby("well_id", sort=False)for col in SENSOR_COLS:    df[f"{col}_roll15_mean"] = grouped[col].transform(lambda s: s.rolling(15, min_periods=1).mean()).astype("float32")    df[f"{col}_roll15_std"] = grouped[col].transform(lambda s: s.rolling(15, min_periods=1).std()).fillna(0).astype("float32")    df[f"{col}_roll60_mean"] = grouped[col].transform(lambda s: s.rolling(60, min_periods=1).mean()).astype("float32")    df[f"{col}_roll60_std"] = grouped[col].transform(lambda s: s.rolling(60, min_periods=1).std()).fillna(0).astype("float32")    df[f"{col}_roll240_mean"] = grouped[col].transform(lambda s: s.rolling(240, min_periods=1).mean()).astype("float32")    df[f"{col}_chg5"] = grouped[col].transform(lambda s: s.diff(5)).fillna(0).astype("float32")    gc.collect()del groupedgc.collect()print("Rolling features done.")

### Relative features - tried, and turned off on purposeAn earlier experiment replaced the absolute features with relative ones (each sensor's deviation from that well's own recent history), on the idea that they'd generalize better to unseen wells. **Documented result**: test performance also dropped (0.66 -> 0.63), while validation performance collapsed (0.82 -> 0.685) and the event-level catch rate fell (49% -> 14%). So the absolute values were carrying real signal, not just memorizing which well is which. The code is kept behind `USE_RELATIVE_FEATURES = False` as documentation of the experiment, not for deletion.

In [ ]:
USE_RELATIVE_FEATURES = FalseEPS = 1e-3REL_FEATURES = []if USE_RELATIVE_FEATURES:    bep_map = specs["pump_bep_hz"].to_dict()    df["freq_vs_bep"] = (df["frequency_hz"] - df["well_id"].map(bep_map)).astype("float32")    REL_FEATURES = ["freq_vs_bep"]    for col in SENSOR_COLS:        base = df[f"{col}_roll240_mean"]        safe_base = base.abs().clip(lower=EPS)        # short- and medium-term deviation from the well's own 4-hour baseline        df[f"{col}_dev15"] = (df[f"{col}_roll15_mean"] - base).astype("float32")        df[f"{col}_dev60"] = (df[f"{col}_roll60_mean"] - base).astype("float32")        # scale-free versions: percentage deviation transfers across wells with        # different absolute operating levels        df[f"{col}_pctdev15"] = ((df[f"{col}_roll15_mean"] - base) / safe_base).astype("float32")        df[f"{col}_pctdev60"] = ((df[f"{col}_roll60_mean"] - base) / safe_base).astype("float32")        # variability relative to level (coefficient of variation)        df[f"{col}_cv15"] = (df[f"{col}_roll15_std"] / safe_base).astype("float32")        df[f"{col}_cv60"] = (df[f"{col}_roll60_std"] / safe_base).astype("float32")        # short-term rate of change, scale-free        df[f"{col}_pctchg5"] = (df[f"{col}_chg5"] / safe_base).astype("float32")        new_cols = [f"{col}_dev15", f"{col}_dev60", f"{col}_pctdev15",                    f"{col}_pctdev60", f"{col}_cv15", f"{col}_cv60", f"{col}_pctchg5"]        # guard against inf/NaN from any residual near-zero denominator, per column        # (doing this once over all 42 columns at the end would materialize a        # second full-width copy and blow the memory budget)        for nc in new_cols:            df[nc] = df[nc].replace([np.inf, -np.inf], 0).fillna(0).astype("float32")        REL_FEATURES += new_cols        gc.collect()print(f"Relative features: {'ON, ' + str(len(REL_FEATURES)) + ' features' if USE_RELATIVE_FEATURES else 'OFF (tested, measurably worse - see header)'}")

### Build the target label: will a shutdown happen in the next H minutes?For each well: compute how many minutes until the nearest upcoming shutdown, then set `label_trip_within_H = 1` if that's within the horizon H. Also builds a "run life" feature (`minutes_since_last_shutdown`) as a cumulative degradation signal that can generalize across wells.

In [ ]:
df["label_trip_within_H"] = np.int8(0)df["minutes_since_last_shutdown"] = np.float32(0)for well_id, idx in df.groupby("well_id", sort=False).groups.items():    shutdown_arr = df.loc[idx, "shutdown_event"].to_numpy()    n = len(shutdown_arr)    next_shutdown_idx = np.full(n, np.inf)    nxt = np.inf    for i in range(n - 1, -1, -1):        if shutdown_arr[i] == 1:            nxt = i        next_shutdown_idx[i] = nxt    minutes_to_shutdown = next_shutdown_idx - np.arange(n)    label = ((minutes_to_shutdown > 0) & (minutes_to_shutdown <= H)).astype("int8")    # "Run life" feature: minutes elapsed since this well's last shutdown    # (or since well start if none yet). Mirrors a standard real-world ESP    # monitoring signal (time since last pull/workover), and stands in for    # the simulator's latent degradation index, which grows roughly    # monotonically between shutdowns - a well-generalizable, physically    # grounded leading indicator instead of relying on noisy sensor levels    # alone. Requested: improve detection while keeping H=60 (the    # operationally required lead time) rather than shortening it further.    since_last = np.empty(n, dtype="float32")    last_idx = None    for i in range(n):        since_last[i] = float(i) if last_idx is None else float(i - last_idx)        if shutdown_arr[i] == 1:            last_idx = i    df.loc[idx, "label_trip_within_H"] = label    df.loc[idx, "minutes_since_last_shutdown"] = since_lastgc.collect()if USE_RELATIVE_FEATURES:    REL_FEATURES.append("minutes_since_last_shutdown")  # well-relative by constructionprint("Labels and run-life feature done.")

### Final data prep: drop unused columns, keep only running rowsThe small shutdown-events table is extracted first, then the heavy text columns (`failure_cause`, `shutdown_event`, `split`) are dropped and the remaining text columns are converted to `category` to save memory; `shutdown` rows (zero frequency) are excluded from training.

In [ ]:
shutdown_events_df = df.loc[df["shutdown_event"] == 1, ["well_id", "timestamp", "failure_cause"]].copy()shutdown_events_df["failure_cause"] = shutdown_events_df["failure_cause"].astype(str)df = df.drop(columns=["failure_cause", "shutdown_event", "split"])df["onset_cause"] = df["onset_cause"].fillna("").astype("category")df["state"] = df["state"].astype("category")gc.collect()mask_running = df["state"].isin(["normal", "onset", "ramp"])df = df[mask_running]          # rebind rather than keeping both frames alivegc.collect()model_df = df.reset_index(drop=True)del dfgc.collect()

### Split the data: train/test wells, then a time split for validationOnly the last 6 weeks of the training wells are used as a time-based validation window, to avoid any time leakage.

In [ ]:
train_pool = model_df[model_df["well_id"].isin(TRAIN_WELLS)]test_pool = model_df[model_df["well_id"].isin(TEST_WELLS)]del model_dfgc.collect()val_cutoff = train_pool["timestamp"].max() - pd.Timedelta(weeks=6)fit_set = train_pool[train_pool["timestamp"] < val_cutoff].copy()val_set = train_pool[train_pool["timestamp"] >= val_cutoff].copy()del train_pool  # fit_set + val_set now hold everything it didgc.collect()print(f"\nFit rows (train wells, early period): {len(fit_set):,}")print(f"Val rows (train wells, final 6 weeks - time-holdout): {len(val_set):,}")print(f"Test rows (fully held-out wells 8,9,10): {len(test_pool):,}")

### Per-well z-score normalizationFor training wells: the baseline (mean/std) is computed from the fit period only. For held-out test wells: the baseline comes from that well's own first 30 days (a realistic "calibration window" after a new well starts producing). Each feature's standard deviation has a floor (20% of the overall standard deviation) to stop z-scores from blowing up when variance is near zero.

In [ ]:
STAT_SAMPLE = 100_000global_std_floor = (fit_set[ALL_FEATURES].sample(n=min(300_000, len(fit_set)), random_state=7).std() * 0.2).replace(0, 1e-3)baseline_stats = {}for well_id in TRAIN_WELLS:    wf = fit_set.loc[fit_set["well_id"] == well_id, ALL_FEATURES]    if len(wf) > STAT_SAMPLE:        wf = wf.sample(n=STAT_SAMPLE, random_state=7)    std = wf.std().combine(global_std_floor, max).replace(0, 1)    baseline_stats[well_id] = (wf.mean(), std)    del wf    gc.collect()for well_id in TEST_WELLS:    wf_all = test_pool[test_pool["well_id"] == well_id]    calib_cutoff = wf_all["timestamp"].min() + pd.Timedelta(days=30)    wf = wf_all.loc[wf_all["timestamp"] < calib_cutoff, ALL_FEATURES]    if len(wf) > STAT_SAMPLE:        wf = wf.sample(n=STAT_SAMPLE, random_state=7)    std = wf.std().combine(global_std_floor, max).replace(0, 1)    baseline_stats[well_id] = (wf.mean(), std)    del wf_all, wf    gc.collect()# Only the columns Stage 2 and the clustering branch actually read - copying# every column here (including Stage 1's 42 relative features, which the# normalized path never touches) is what pushed this past the memory budget.NORM_KEEP = ["well_id", "timestamp", "onset_cause", "label_trip_within_H"]def normalize(frame):    out = frame[NORM_KEEP + ALL_FEATURES].copy()    for well_id, g in frame.groupby("well_id"):        mean, std = baseline_stats[well_id]        idx = g.index        out.loc[idx, ALL_FEATURES] = ((g[ALL_FEATURES] - mean) / std).astype("float32").values    return out

### Apply normalization only to the samples that need it (saves memory)Normalization is applied right after sampling/subsetting - not on the full 2.6-million-row table - because the clustering branch only needs a small sample, and Stage 2 only needs the true `onset` rows.

In [ ]:
clust_sample = normalize(fit_set.sample(n=min(150_000, len(fit_set)), random_state=7))cause_fit = normalize(fit_set[fit_set["onset_cause"].astype(str).isin(CAUSE_NAMES)])cause_test = normalize(test_pool[test_pool["onset_cause"].astype(str).isin(CAUSE_NAMES)])gc.collect()if USE_RELATIVE_FEATURES:    STAGE1_KEEP = ["well_id", "timestamp", "label_trip_within_H"] + REL_FEATURES    DROP_COLS = [c for c in ALL_FEATURES if c not in STAGE1_KEEP]    # keep the two raw columns the SCADA-rule baseline and the timeline chart read    DROP_COLS = [c for c in DROP_COLS                 if c not in ("vibration_g", "vibration_g_roll15_mean", "motor_current_a_roll15_mean")]    fit_set = fit_set.drop(columns=DROP_COLS)    val_set = val_set.drop(columns=DROP_COLS)    test_pool = test_pool.drop(columns=DROP_COLS)    gc.collect()    print(f"Dropped {len(DROP_COLS)} absolute-level columns from the Stage 1 frames "          f"(kept by the small normalized subsets above).")

### Bonus chart - Before & After: why per-well normalization mattersOne feature (`vibration_g_roll15_mean`), one box per well, shown twice:- **BEFORE**: the raw value, straight from the sensor. Each well sits at a different baseline level and spread, because each well has its own equipment and operating point.- **AFTER**: the same feature once it has been z-score normalized *per well*. All wells should now be centered near 0 with comparable spread.This is the visual proof for the normalization design choice explained earlier: without it, a model trained on one well's absolute sensor levels would not transfer to a well it has never seen.

In [ ]:
feat = "vibration_g_roll15_mean"
wells_sorted = sorted(fit_set["well_id"].unique())

rng_plot = np.random.default_rng(7)

def sample_per_well(frame, col, wells, n=3000):
    out = []
    for w in wells:
        vals = frame.loc[frame["well_id"] == w, col].dropna().to_numpy()
        if len(vals) > n:
            idx = rng_plot.choice(len(vals), size=n, replace=False)
            vals = vals[idx]
        out.append(vals)
    return out

data_before = sample_per_well(fit_set, feat, wells_sorted)      # raw, not normalized
data_after  = sample_per_well(clust_sample, feat, wells_sorted)  # already z-scored per well

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=False)

axes[0].boxplot(data_before, tick_labels=[str(w) for w in wells_sorted], showfliers=False)
axes[0].set_title(f"BEFORE: raw {feat}\n(one box per well)")
axes[0].set_xlabel("well_id"); axes[0].set_ylabel(feat)
axes[0].grid(axis="y", alpha=0.25, linewidth=0.6)

axes[1].axhline(0, color="gray", linestyle=":", linewidth=1)
axes[1].boxplot(data_after, tick_labels=[str(w) for w in wells_sorted], showfliers=False)
axes[1].set_title(f"AFTER: per-well z-scored {feat}\n(one box per well)")
axes[1].set_xlabel("well_id"); axes[1].set_ylabel(f"{feat} (z-score)")
axes[1].grid(axis="y", alpha=0.25, linewidth=0.6)

plt.suptitle("Per-well normalization removes the well-to-well baseline gap")
plt.tight_layout()
plt.savefig(f"{OUT}/before_after_normalization.png", dpi=150)
plt.close()
print("Saved before_after_normalization.png")

### Unsupervised branch - K-MeansClusters operating states into 6 groups on the normalized features, then links each cluster to its failure rate.

**Before & After: class separability (raw vs. normalized)**  `before_after_separability.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/before_after_separability.png'))

**Before & After the trip: precursor vs. abrupt failure**  `before_after_precursor.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/before_after_precursor.png'))

**Before & After: naive threshold vs. tuned alarm policy**  `before_after_alarm_policy.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/before_after_alarm_policy.png'))

**Before & After: 7 vs. 13 training wells**  `before_after_more_wells.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/before_after_more_wells.png'))

In [ ]:
print("\n=== UNSUPERVISED: Clustering operating states ===")Xc = clust_sample[ALL_FEATURES].to_numpy(dtype="float32")km = KMeans(n_clusters=6, n_init=10, random_state=7).fit(Xc)km_sil = silhouette_score(Xc, km.labels_, sample_size=20_000, random_state=7)print(f"K-Means (k=6) silhouette: {km_sil:.3f}")clust_sample = clust_sample.copy()clust_sample["kmeans_cluster"] = km.labels_cluster_failure_rate = clust_sample.groupby("kmeans_cluster")["label_trip_within_H"].mean()print("K-Means cluster -> failure-within-H rate:")print(cluster_failure_rate.to_string())

### Unsupervised branch - DBSCANCompares the failure rate between outlier points and points inside a normal cluster.

In [ ]:
db_sample = clust_sample.sample(n=min(40_000, len(clust_sample)), random_state=7)Xdb = db_sample[ALL_FEATURES].to_numpy(dtype="float32")db = DBSCAN(eps=2.0, min_samples=20).fit(Xdb)n_db_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)db_sample = db_sample.copy()db_sample["dbscan_cluster"] = db.labels_outlier_failure_rate = db_sample.groupby(db_sample["dbscan_cluster"] == -1)["label_trip_within_H"].mean()print(f"\nDBSCAN found {n_db_clusters} clusters (+ noise/outliers)")print(f"Failure-within-H rate: outliers={outlier_failure_rate.get(True, float('nan')):.4f} "      f"vs. clustered-normal={outlier_failure_rate.get(False, float('nan')):.4f}")

### PCA plot: K-Means clusters vs. the actual label, on the same projection

In [ ]:
pca = PCA(n_components=2, random_state=7).fit(Xc)Xc_pca = pca.transform(Xc)fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))sc0 = axes[0].scatter(Xc_pca[:, 0], Xc_pca[:, 1], c=km.labels_, cmap="tab10", s=3, alpha=0.4)axes[0].set_title("K-Means clusters (PCA, well-normalized features)")axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")plt.colorbar(sc0, ax=axes[0], label="cluster")sc1 = axes[1].scatter(Xc_pca[:, 0], Xc_pca[:, 1], c=clust_sample["label_trip_within_H"], cmap="coolwarm", s=3, alpha=0.4)axes[1].set_title("Actual failure-within-H label (same projection)")axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")plt.colorbar(sc1, ax=axes[1], label="trip within H")plt.tight_layout()plt.savefig(f"{OUT}/clustering_pca_normalized.png", dpi=150)plt.close()print("Saved clustering_pca_normalized.png")del clust_sample, db_sample, Xc, Xdb, Xc_pcagc.collect()

### Stage 1 (failure detection) - prepare train/validation/test setsThe negative rows (no failure coming soon) are downsampled to 40 negatives per positive, because failures are rare compared to normal running time.

In [ ]:
STAGE1_FEATURES = REL_FEATURES if USE_RELATIVE_FEATURES else ALL_FEATURESNEG_PER_POS = 40pos_fit = fit_set[fit_set["label_trip_within_H"] == 1]neg_fit_full = fit_set[fit_set["label_trip_within_H"] == 0]n_neg_keep = min(len(neg_fit_full), len(pos_fit) * NEG_PER_POS)neg_fit = neg_fit_full.sample(n=n_neg_keep, random_state=7)fit_ds = pd.concat([pos_fit, neg_fit]).sort_values("timestamp")print(f"\nFit rows after downsampling (train-only, {len(STAGE1_FEATURES)} features): {len(fit_ds):,} "      f"({len(pos_fit):,} positive + {len(neg_fit):,} negative, ratio 1:{NEG_PER_POS})")Xfit = fit_ds[STAGE1_FEATURES].to_numpy(dtype="float32")yfit = fit_ds["label_trip_within_H"].to_numpy()Xval = val_set[STAGE1_FEATURES].to_numpy(dtype="float32")yval = val_set["label_trip_within_H"].to_numpy()Xtest = test_pool[STAGE1_FEATURES].to_numpy(dtype="float32")ytest = test_pool["label_trip_within_H"].to_numpy()del pos_fit, neg_fit_full, neg_fitgc.collect()

### Bonus chart - Before & After: does normalization actually separate the classes better?Same idea as the earlier before/after chart, but now split by outcome (normal vs. a trip within H) instead of by well, and pooled across ALL training wells together.- **BEFORE**: the raw feature, pooled across wells - the two classes should overlap heavily, because the between-well differences swamp the failure signal.- **AFTER**: the same rows once normalized per well - the "trip" class should separate more cleanly from "normal".**Honest note**: Stage 1 in this notebook is actually trained on `STAGE1_FEATURES = ALL_FEATURES`, i.e. the **raw**, non-normalized version (see the code two cells above) - only the unsupervised branch and Stage 2 use the normalized features. This chart shows what normalizing Stage 1's own rows *would* look like, as an honest check on whether that's worth doing - it does not claim this is what Stage 1 currently sees.

In [ ]:
feat = "vibration_g_roll15_mean"

before_pos = fit_ds.loc[fit_ds["label_trip_within_H"] == 1, feat].to_numpy()
before_neg = fit_ds.loc[fit_ds["label_trip_within_H"] == 0, feat].to_numpy()

fit_ds_norm = normalize(fit_ds)
after_pos = fit_ds_norm.loc[fit_ds_norm["label_trip_within_H"] == 1, feat].to_numpy()
after_neg = fit_ds_norm.loc[fit_ds_norm["label_trip_within_H"] == 0, feat].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))

axes[0].hist(before_neg, bins=60, density=True, alpha=0.55, label="normal", color="#4C72B0")
axes[0].hist(before_pos, bins=60, density=True, alpha=0.55, label="trip within H", color="#C1666B")
axes[0].set_title(f"BEFORE: raw {feat}\n(all wells pooled)")
axes[0].set_xlabel(feat); axes[0].set_ylabel("density"); axes[0].legend(fontsize=8)

axes[1].hist(after_neg, bins=60, density=True, alpha=0.55, label="normal", color="#4C72B0")
axes[1].hist(after_pos, bins=60, density=True, alpha=0.55, label="trip within H", color="#C1666B")
axes[1].set_title(f"AFTER: per-well z-scored {feat}\n(all wells pooled)")
axes[1].set_xlabel(f"{feat} (z-score)"); axes[1].set_ylabel("density"); axes[1].legend(fontsize=8)

plt.suptitle("Class separability, raw vs. per-well normalized (pooled across wells)")
plt.tight_layout()
plt.savefig(f"{OUT}/before_after_separability.png", dpi=150)
plt.close()
print("Saved before_after_separability.png")

### Define and train the three modelsLogistic Regression, Random Forest, and XGBoost - each evaluated with ROC-AUC on both the validation and test sets.

In [ ]:
models = {    "Logistic Regression": LogisticRegression(max_iter=3000, class_weight="balanced"),    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=9, class_weight="balanced_subsample", random_state=7, n_jobs=2),    "XGBoost": XGBClassifier(        n_estimators=150, max_depth=5, learning_rate=0.08, subsample=0.9, colsample_bytree=0.9,        eval_metric="logloss", scale_pos_weight=(yfit == 0).sum() / max((yfit == 1).sum(), 1),        random_state=7, n_jobs=2,    ),}results = []test_probas = {}val_probas = {}print("\n=== SUPERVISED STAGE 1: Failure detection ===")for name, model in models.items():    print(f"  training {name}...")    model.fit(Xfit, yfit)    val_proba = model.predict_proba(Xval)[:, 1]    test_proba = model.predict_proba(Xtest)[:, 1]    val_probas[name] = val_proba    test_pred = (test_proba >= 0.5).astype(int)    results.append({        "Model": name,        "Val ROC-AUC": roc_auc_score(yval, val_proba),        "Test Accuracy": accuracy_score(ytest, test_pred),        "Test Precision": precision_score(ytest, test_pred, zero_division=0),        "Test Recall": recall_score(ytest, test_pred, zero_division=0),        "Test F1": f1_score(ytest, test_pred, zero_division=0),        "Test ROC-AUC": roc_auc_score(ytest, test_proba),    })    test_probas[name] = test_proba    gc.collect()

### Baselines: a fixed-threshold rule, and last-value/persistence

In [ ]:
vib_thresh = fit_set["vibration_g_roll15_mean"].mean() + 2.5 * fit_set["vibration_g_roll15_mean"].std()cur_thresh_low = fit_set["motor_current_a_roll15_mean"].mean() - 2.5 * fit_set["motor_current_a_roll15_mean"].std()rule_pred = ((test_pool["vibration_g_roll15_mean"] > vib_thresh) |             (test_pool["motor_current_a_roll15_mean"] < cur_thresh_low)).astype(int).valuesresults.append({    "Model": "Baseline: fixed-threshold rule", "Val ROC-AUC": np.nan,    "Test Accuracy": accuracy_score(ytest, rule_pred),    "Test Precision": precision_score(ytest, rule_pred, zero_division=0),    "Test Recall": recall_score(ytest, rule_pred, zero_division=0),    "Test F1": f1_score(ytest, rule_pred, zero_division=0), "Test ROC-AUC": np.nan,})persistence_pred = test_pool.groupby("well_id")["label_trip_within_H"].shift(1).fillna(0).astype(int).valuesresults.append({    "Model": "Baseline: persistence (last value)", "Val ROC-AUC": np.nan,    "Test Accuracy": accuracy_score(ytest, persistence_pred),    "Test Precision": precision_score(ytest, persistence_pred, zero_division=0),    "Test Recall": recall_score(ytest, persistence_pred, zero_division=0),    "Test F1": f1_score(ytest, persistence_pred, zero_division=0), "Test ROC-AUC": np.nan,})

### Show the Stage 1 results table and pick the best modelThe choice is based on **validation ROC-AUC only**, never on test, to avoid any form of leakage from picking a model based on data it's supposed to have never seen.

In [ ]:
results_df = pd.DataFrame(results)print("\n=== STAGE 1 results: models vs. baselines (held-out wells 8,9,10) ===")print(results_df.to_string(index=False))# Model selection uses VALIDATION ROC-AUC, never test. A previous version of# this script selected on Test ROC-AUC, which is a (mild but real) form of# the leakage the instructor explicitly asked us to prevent - the held-out# wells must not influence any modeling decision, including which model# gets shipped. Fixed here; it does change which model wins.best_name = results_df[results_df["Model"].isin(models.keys())].sort_values("Val ROC-AUC", ascending=False).iloc[0]["Model"]best_proba = test_probas[best_name]best_val_proba = val_probas[best_name]print(f"\nBest model (selected on VALIDATION ROC-AUC): {best_name}")

### Alarm policy - helper functionsThree functions: `per_well_thresholds` (a per-well relative alert cutoff), `alarm_events` (turns probability scores into connected alarm events after requiring N minutes of persistence), and `score_policy` (scores a full alarm policy: how many events were caught vs. missed, false alarms, and lead time).

In [ ]:
MISS_COST_RATIO = 200.0ALERT_WINDOW_MIN = 180   # an alarm counts as "catching" a trip if it fires within this window before itdef per_well_thresholds(frame, proba_col, q):    """Alert cutoff as a quantile of EACH WELL's own score distribution.    A single absolute probability threshold does not transfer between wells:    tuned on validation, 0.90 was the 99.9th percentile there, but on the    held-out wells the same model's scores sit far lower, so that threshold    fired twice in six months. Each well gets its own cutoff from its own    score distribution instead - "alert on this well's riskiest X% of    minutes" is a rule that means the same thing everywhere, and it needs no    labels from the new well, only its own history.    """    return frame.groupby("well_id")[proba_col].quantile(q).to_dict()def alarm_events(frame, proba_col, thresh_map, n_persist):    """Return alarm start/end timestamps per well after persistence filtering."""    events = []    for well_id, g in frame.groupby("well_id", sort=False):        g = g.sort_values("timestamp")        above = (g[proba_col].to_numpy() >= thresh_map[well_id]).astype(int)        if n_persist > 1:            # rolling sum over the last n_persist minutes must be == n_persist            csum = np.convolve(above, np.ones(n_persist, dtype=int), mode="full")[:len(above)]            fired = (csum == n_persist).astype(int)        else:            fired = above        ts = g["timestamp"].to_numpy()        # group consecutive fired minutes into single alarm events (vectorized:        # a +1 in the diff marks a run start, a -1 marks one past the run end)        d = np.diff(np.concatenate(([0], fired, [0])))        starts = np.where(d == 1)[0]        ends = np.where(d == -1)[0] - 1        if len(starts):            events.append(pd.DataFrame({"well_id": well_id, "start": ts[starts], "end": ts[ends]}))    return pd.concat(events, ignore_index=True) if events else pd.DataFrame(columns=["well_id", "start", "end"])def score_policy(frame, proba_col, trips_df, q, n_persist):    """Evaluate an alarm policy: returns caught/missed trips, false-alarm events, lead times.    An alarm counts as catching a trip if it is ACTIVE at any point in the    alert window before that trip - i.e. the alarm interval OVERLAPS the    window. Requiring the alarm to *start* inside the window (a first    version of this did) silently scores a continuously-firing alarm as    catching nothing, since its start sits far in the past.    Lead time is measured from when the alarm was already active at the    window edge, not from the alarm's own (possibly much earlier) start.    """    ev = alarm_events(frame, proba_col, per_well_thresholds(frame, proba_col, q), n_persist)    caught, missed, leads = 0, 0, []    matched_alarm_idx = set()    for _, trip in trips_df.iterrows():        st, wid = trip["timestamp"], trip["well_id"]        if len(ev) == 0:            missed += 1            continue        window_start = st - pd.Timedelta(minutes=ALERT_WINDOW_MIN)        # overlap test: alarm starts before the trip AND ends after the window opens        hits = ev[(ev["well_id"] == wid) & (ev["start"] < st) & (ev["end"] >= window_start)]        if len(hits) > 0:            caught += 1            first_active = max(hits["start"].min(), window_start)            leads.append((st - first_active).total_seconds() / 60.0)            matched_alarm_idx.update(hits.index.tolist())        else:            missed += 1    false_alarms = 0 if len(ev) == 0 else len(ev) - len(matched_alarm_idx)    return {"caught": caught, "missed": missed, "false_alarm_events": false_alarms,            "leads": leads, "n_alarms": len(ev)}

### Tune the alarm policy on validation data onlyA grid search over (relative threshold x persistence minutes), picking the lowest-cost combination, where missing one failure costs 200x as much as one false alarm (`MISS_COST_RATIO`).

In [ ]:
val_set = val_set.reset_index(drop=True)val_set["proba"] = best_val_probaval_trips = shutdown_events_df[    shutdown_events_df["well_id"].isin(TRAIN_WELLS) &    (shutdown_events_df["timestamp"] >= val_set["timestamp"].min())]print(f"\nTuning alarm policy on validation ({len(val_trips)} trips in the validation window)...")# Thresholds are taken as QUANTILES of the model's own validation scores, not# fixed absolute values. Different models put their probabilities on wildly# different scales here - class-balanced Logistic Regression pushes most rows# above 0.5 on a 1%-positive problem, while the trees rarely exceed 0.3 - so a# hard-coded absolute grid fits one model and completely misses the other.# Quantiles adapt to whatever scale the selected model produces.policy_grid = []QUANTILES = [0.950, 0.975, 0.990, 0.995, 0.999]for n_persist in [1, 3, 5, 10, 15, 30]:    for q in QUANTILES:        r = score_policy(val_set, "proba", val_trips, q, n_persist)        cost = r["missed"] * MISS_COST_RATIO + r["false_alarm_events"]        policy_grid.append({"n_persist": n_persist, "quantile": q, "caught": r["caught"],                             "missed": r["missed"], "false_alarms": r["false_alarm_events"], "cost": cost})policy_df = pd.DataFrame(policy_grid).sort_values("cost")print(policy_df.head(10).to_string(index=False))best_policy = policy_df.iloc[0]Q = float(best_policy["quantile"])N_PERSIST = int(best_policy["n_persist"])print(f"\nSelected alarm policy (validation, cost-weighted at {MISS_COST_RATIO:.0f}:1): "      f"per-well top {100*(1-Q):.1f}% of scores, persistence={N_PERSIST} consecutive minutes")

### Apply the chosen policy to the fully held-out test wells, and compute final performance

In [ ]:
test_pool = test_pool.reset_index(drop=True)test_pool["proba"] = best_probatest_thresh_map = per_well_thresholds(test_pool, "proba", Q)test_pool["pred"] = (test_pool["proba"] >= test_pool["well_id"].map(test_thresh_map)).astype(int)print("Per-well alert thresholds on held-out wells: "      + ", ".join(f"well {w}: {t:.4f}" for w, t in sorted(test_thresh_map.items())))cm = confusion_matrix(ytest, test_pool["pred"])tn, fp, fn, tp = cm.ravel()print(f"\nMinute-level confusion matrix (before persistence filter): TN={tn} FP={fp} FN={fn} TP={tp}")test_trips = shutdown_events_df[shutdown_events_df["well_id"].isin(TEST_WELLS)]test_trips = test_trips[test_trips["timestamp"] >= test_pool["timestamp"].min()]final = score_policy(test_pool, "proba", test_trips, Q, N_PERSIST)caught_events, missed_events = final["caught"], final["missed"]lead_times = final["leads"]false_alarm_events = final["false_alarm_events"]n_events = caught_events + missed_events# how many operator-facing alarms per well per month, a number an engineer# can actually reason abouttest_days = (test_pool["timestamp"].max() - test_pool["timestamp"].min()).total_seconds() / 86400fa_per_well_month = false_alarm_events / max(len(TEST_WELLS), 1) / max(test_days / 30.0, 1e-9)event_log = []for well_id in TEST_WELLS:    wt = test_trips[test_trips["well_id"] == well_id]    for _, ev_row in wt.iterrows():        event_log.append({"well_id": well_id, "timestamp": ev_row["timestamp"],                           "cause": ev_row["failure_cause"], "caught": True, "lead_min": None})print(f"\nTrips in held-out wells: {n_events}")print(f"Caught (>=1 alarm in prior {ALERT_WINDOW_MIN} min): {caught_events} ({100*caught_events/max(n_events,1):.0f}%)")print(f"Missed failures: {missed_events} ({100*missed_events/max(n_events,1):.0f}%)")print(f"False-alarm EVENTS: {false_alarm_events} (~{fa_per_well_month:.1f} per well per month)")print(f"Total alarms raised: {final['n_alarms']}")if lead_times:    print(f"Average lead time: {np.mean(lead_times):.1f} min (median {np.median(lead_times):.1f} min)")

### Bonus chart - Before & After: naive fixed threshold vs. the tuned alarm policy- **BEFORE (naive)**: alert whenever the model's probability crosses a flat 0.5, with no persistence requirement - roughly what a less careful deployment might ship.- **AFTER (tuned)**: the per-well quantile threshold plus persistence filter, tuned on validation only, that this notebook actually ships.This turns the numbers already printed earlier in Step 4 into a direct visual comparison: same model, same test wells, only the alarm policy changes.

In [ ]:
def score_policy_flat(frame, proba_col, trips_df, thresh_map, n_persist):
    """Same scoring as score_policy(), but takes a ready-made threshold map instead of a
    validation-tuned quantile. Used here only to score a naive flat 0.5 cutoff for
    comparison - the alarm policy actually shipped is the one tuned by score_policy()
    elsewhere in this notebook."""
    ev = alarm_events(frame, proba_col, thresh_map, n_persist)
    caught, missed, leads = 0, 0, []
    matched_alarm_idx = set()
    for _, trip in trips_df.iterrows():
        st, wid = trip["timestamp"], trip["well_id"]
        if len(ev) == 0:
            missed += 1
            continue
        window_start = st - pd.Timedelta(minutes=ALERT_WINDOW_MIN)
        hits = ev[(ev["well_id"] == wid) & (ev["start"] < st) & (ev["end"] >= window_start)]
        if len(hits) > 0:
            caught += 1
            first_active = max(hits["start"].min(), window_start)
            leads.append((st - first_active).total_seconds() / 60.0)
            matched_alarm_idx.update(hits.index.tolist())
        else:
            missed += 1
    false_alarms = 0 if len(ev) == 0 else len(ev) - len(matched_alarm_idx)
    return {"caught": caught, "missed": missed, "false_alarm_events": false_alarms,
            "leads": leads, "n_alarms": len(ev)}

naive_thresh_map = {w: 0.5 for w in TEST_WELLS}
naive = score_policy_flat(test_pool, "proba", test_trips, naive_thresh_map, n_persist=1)

n_events_naive = naive["caught"] + naive["missed"]
naive_catch_pct = 100 * naive["caught"] / max(n_events_naive, 1)
naive_fa_per_well_month = naive["false_alarm_events"] / max(len(TEST_WELLS), 1) / max(test_days / 30.0, 1e-6)
naive_lead = float(np.mean(naive["leads"])) if naive["leads"] else 0.0

tuned_catch_pct = 100 * caught_events / max(n_events, 1)
tuned_lead = float(np.mean(lead_times)) if lead_times else 0.0

labels = ["Naive: fixed 0.5\nthreshold, no persistence", "Tuned: per-well quantile\n+ persistence (shipped)"]
catch_vals = [naive_catch_pct, tuned_catch_pct]
fa_vals = [naive_fa_per_well_month, fa_per_well_month]
lead_vals = [naive_lead, tuned_lead]

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.9))
colors = ["#C1666B", "#2A9D8F"]
axes[0].bar(labels, catch_vals, color=colors); axes[0].set_title("Trips caught (%)")
axes[1].bar(labels, fa_vals, color=colors); axes[1].set_title("False alarms\n(per well per month)")
axes[2].bar(labels, lead_vals, color=colors); axes[2].set_title("Average lead time (min)")
for ax in axes:
    ax.tick_params(axis="x", labelsize=7.5)
plt.suptitle("Before & After: a naive fixed threshold vs. the validation-tuned alarm policy")
plt.tight_layout()
plt.savefig(f"{OUT}/before_after_alarm_policy.png", dpi=150)
plt.close()
print("Saved before_after_alarm_policy.png")

### Full trade-off curve on the test wells - for reporting onlyThis table is not used to pick the shipped policy; it just shows what could have been achieved with different cost preferences.

In [ ]:
tradeoff_rows = []for n_p in [1, 3, 5, 10]:    for q in QUANTILES:        r = score_policy(test_pool, "proba", test_trips, q, n_p)        n_ev = r["caught"] + r["missed"]        tradeoff_rows.append({            "persist_min": n_p, "top_pct_of_scores": round(100 * (1 - q), 2),            "caught": r["caught"], "caught_pct": round(100 * r["caught"] / max(n_ev, 1)),            "missed": r["missed"], "false_alarm_events": r["false_alarm_events"],            "fa_per_well_month": round(r["false_alarm_events"] / max(len(TEST_WELLS), 1) / max(test_days / 30.0, 1e-9), 1),            "avg_lead_min": round(float(np.mean(r["leads"])), 1) if r["leads"] else None,        })tradeoff_df = pd.DataFrame(tradeoff_rows).sort_values(["persist_min", "top_pct_of_scores"])print("\n=== Trade-off curve on held-out wells (reported, NOT used for selection) ===")print(tradeoff_df.to_string(index=False))

### Evaluate each model with its own tuned policyThe earlier table compared models at a fixed 0.5 threshold (not realistic in operation); here every model gets its own properly-tuned alarm policy, then is measured on real events.

In [ ]:
print("\n=== PER-MODEL event-level performance on held-out wells ===")per_model_rows = []per_model_policy = {}for mname in models.keys():    val_set["proba_m"] = val_probas[mname]    test_pool["proba_m"] = test_probas[mname]    # tune this model's own policy on validation    grid = []    for n_p in [1, 3, 5, 10, 15, 30]:        for q in QUANTILES:            r = score_policy(val_set, "proba_m", val_trips, q, n_p)            grid.append((r["missed"] * MISS_COST_RATIO + r["false_alarm_events"], q, n_p))    grid.sort()    _, q_best, n_best = grid[0]    per_model_policy[mname] = (q_best, n_best)    r = score_policy(test_pool, "proba_m", test_trips, q_best, n_best)    n_ev = r["caught"] + r["missed"]    per_model_rows.append({        "Model": mname,        "Val ROC-AUC": round(roc_auc_score(yval, val_probas[mname]), 3),        "Test ROC-AUC": round(roc_auc_score(ytest, test_probas[mname]), 3),        "policy_top_pct": round(100 * (1 - q_best), 2),        "policy_persist_min": n_best,        "caught": r["caught"],        "caught_pct": round(100 * r["caught"] / max(n_ev, 1)),        "missed": r["missed"],        "nuisance_alarms": r["false_alarm_events"],        "fa_per_well_month": round(r["false_alarm_events"] / max(len(TEST_WELLS), 1) / max(test_days / 30.0, 1e-9), 1),        "avg_lead_min": round(float(np.mean(r["leads"])), 1) if r["leads"] else None,        "median_lead_min": round(float(np.median(r["leads"])), 1) if r["leads"] else None,    })per_model_df = pd.DataFrame(per_model_rows)print(per_model_df.to_string(index=False))

### Per-well performance breakdown, for the chosen model

In [ ]:
print("\n=== PER-WELL breakdown (selected model, shipped policy) ===")test_pool["proba_m"] = best_probaper_well_rows = []for wid in TEST_WELLS:    w_frame = test_pool[test_pool["well_id"] == wid]    w_trips = test_trips[test_trips["well_id"] == wid]    r = score_policy(w_frame, "proba_m", w_trips, Q, N_PERSIST)    n_ev = r["caught"] + r["missed"]    spec_row = specs.loc[wid]    per_well_rows.append({        "well": wid,        "dominant_causes": spec_row["dominant_causes"],        "trips": n_ev,        "caught": r["caught"],        "caught_pct": round(100 * r["caught"] / max(n_ev, 1)),        "missed": r["missed"],        "nuisance_alarms": r["false_alarm_events"],        "fa_per_month": round(r["false_alarm_events"] / max(test_days / 30.0, 1e-9), 1),        "avg_lead_min": round(float(np.mean(r["leads"])), 1) if r["leads"] else None,    })per_well_df = pd.DataFrame(per_well_rows)print(per_well_df.to_string(index=False))

### Catch rate by failure cause

In [ ]:
print("\n=== Catch rate by failure cause (selected model) ===")ev_sel = alarm_events(test_pool, "proba_m", per_well_thresholds(test_pool, "proba_m", Q), N_PERSIST)cause_rows = []for _, trip in test_trips.iterrows():    st, wid, cz = trip["timestamp"], trip["well_id"], trip["failure_cause"]    ws = st - pd.Timedelta(minutes=ALERT_WINDOW_MIN)    hit = len(ev_sel[(ev_sel["well_id"] == wid) & (ev_sel["start"] < st) & (ev_sel["end"] >= ws)]) > 0    cause_rows.append({"cause": cz, "caught": int(hit)})cause_catch = pd.DataFrame(cause_rows).groupby("cause")["caught"].agg(["sum", "count"])cause_catch.columns = ["caught", "trips"]cause_catch["caught_pct"] = (100 * cause_catch["caught"] / cause_catch["trips"]).round(0)print(cause_catch.to_string())

### Bonus chart - Before & After: does adding more training wells actually help?The docstring for this step claims that growing the roster from 10 wells to 16 wells (giving every cause 2 training wells instead of 1) improves cross-well generalization. Here that claim is tested directly instead of just asserted:- **BEFORE**: a fresh Random Forest trained only on the original 7 training wells (wells 1-7 - the "10-well" roster, once wells 8/9/10 as test are excluded).- **AFTER**: the same model type trained on all 13 training wells (the current "16-well" roster).Both are evaluated on the exact same 3 fully held-out test wells (8, 9, 10), so this is a fair apples-to-apples comparison, not a re-run with different test data.

In [ ]:
ORIGINAL_TRAIN_WELLS = [w for w in TRAIN_WELLS if w <= 7]
fit_small = fit_ds[fit_ds["well_id"].isin(ORIGINAL_TRAIN_WELLS)]
print(f"'10-well' subset: {len(ORIGINAL_TRAIN_WELLS)} training wells {sorted(ORIGINAL_TRAIN_WELLS)}, {len(fit_small):,} rows")
print(f"'16-well' full set: {len(TRAIN_WELLS)} training wells, {len(fit_ds):,} rows")

Xfit_small = fit_small[STAGE1_FEATURES].to_numpy(dtype="float32")
yfit_small = fit_small["label_trip_within_H"].to_numpy()

rf_small = RandomForestClassifier(n_estimators=150, max_depth=9,
                                   class_weight="balanced_subsample", random_state=7, n_jobs=2)
rf_small.fit(Xfit_small, yfit_small)

proba_small_test = rf_small.predict_proba(Xtest)[:, 1]
auc_small = roc_auc_score(ytest, proba_small_test)
auc_full = roc_auc_score(ytest, test_probas["Random Forest"])

fig, ax = plt.subplots(figsize=(5.5, 4.2))
bar_labels = [f"7 training wells\n('10-well' set)", f"{len(TRAIN_WELLS)} training wells\n('16-well' set)"]
bars = ax.bar(bar_labels, [auc_small, auc_full], color=["#C1666B", "#2A9D8F"])
for b, v in zip(bars, [auc_small, auc_full]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
ax.set_ylabel("ROC-AUC on the same 3 held-out test wells")
ax.set_ylim(0, 1.05)
ax.set_title("Before & After: more training wells,\nsame Random Forest, same test wells")
plt.tight_layout()
plt.savefig(f"{OUT}/before_after_more_wells.png", dpi=150)
plt.close()
print("Saved before_after_more_wells.png")

### Chart 1 of 10 - ROC and Precision-Recall curves on the test wells

In [ ]:
from sklearn.metrics import roc_curve as _roc_curve, precision_recall_curvePALETTE = {"Logistic Regression": "#4C72B0", "Random Forest": "#C96A16", "XGBoost": "#2A9D8F"}GRID_KW = dict(alpha=0.25, linewidth=0.6)# --- 1. ROC + Precision-Recall curves on the held-out wells ---fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))for mname in models.keys():    fpr_c, tpr_c, _ = _roc_curve(ytest, test_probas[mname])    axes[0].plot(fpr_c, tpr_c, label=f"{mname} (AUC={roc_auc_score(ytest, test_probas[mname]):.3f})",                 color=PALETTE[mname], linewidth=1.6)    prec_c, rec_c, _ = precision_recall_curve(ytest, test_probas[mname])    axes[1].plot(rec_c, prec_c, label=mname, color=PALETTE[mname], linewidth=1.6)axes[0].plot([0, 1], [0, 1], color="gray", linestyle=":", linewidth=1, label="random")axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")axes[0].set_title("ROC - held-out wells 8, 9, 10"); axes[0].legend(fontsize=8); axes[0].grid(**GRID_KW)axes[1].axhline(ytest.mean(), color="gray", linestyle=":", linewidth=1,                label=f"base rate ({100*ytest.mean():.2f}%)")axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")axes[1].set_title("Precision-Recall (the honest view for rare events)")axes[1].legend(fontsize=8); axes[1].grid(**GRID_KW); axes[1].set_ylim(0, max(0.05, float(np.nanmax(prec_c[:-1])) * 1.1))plt.tight_layout(); plt.savefig(f"{OUT}/roc_pr_curves.png", dpi=150); plt.close()print("Saved roc_pr_curves.png")

### Chart 2 of 10 - model comparison at the event level (catch rate, false alarms, lead time)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))mnames = per_model_df["Model"].tolist()colors = [PALETTE[m] for m in mnames]axes[0].bar(mnames, per_model_df["caught_pct"], color=colors)axes[0].set_title("Trips caught (%)"); axes[0].set_ylabel("% of 110 trips")for i, v in enumerate(per_model_df["caught_pct"]):    axes[0].text(i, v + 1, f"{v}%", ha="center", fontsize=9)axes[1].bar(mnames, per_model_df["fa_per_well_month"], color=colors)axes[1].set_title("Nuisance alarms per well per month")for i, v in enumerate(per_model_df["fa_per_well_month"]):    axes[1].text(i, v * 1.02, f"{v:.0f}", ha="center", fontsize=9)leads = per_model_df["avg_lead_min"].fillna(0)axes[2].bar(mnames, leads, color=colors)axes[2].set_title("Average lead time (min)")for i, v in enumerate(leads):    axes[2].text(i, v + 2, f"{v:.0f}", ha="center", fontsize=9)for ax in axes:    ax.tick_params(axis="x", rotation=18, labelsize=8); ax.grid(axis="y", **GRID_KW)plt.suptitle("Each model at its OWN validation-tuned alarm policy - held-out wells", fontsize=10)plt.tight_layout(); plt.savefig(f"{OUT}/model_comparison_events.png", dpi=150); plt.close()print("Saved model_comparison_events.png")

### Chart 3 of 10 - trade-off curves for each model

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.6))for mname in models.keys():    test_pool["proba_m"] = test_probas[mname]    xs, ys = [], []    for q in QUANTILES:        r = score_policy(test_pool, "proba_m", test_trips, q, 5)        n_ev = max(r["caught"] + r["missed"], 1)        xs.append(r["false_alarm_events"] / max(len(TEST_WELLS), 1) / max(test_days / 30.0, 1e-9))        ys.append(100 * r["caught"] / n_ev)    ax.plot(xs, ys, "o-", label=mname, color=PALETTE[mname], linewidth=1.6, markersize=4)ax.set_xlabel("Nuisance alarms per well per month"); ax.set_ylabel("Trips caught (%)")ax.set_title("Operating-point trade-off (5-minute persistence)")ax.legend(fontsize=8); ax.grid(**GRID_KW)plt.tight_layout(); plt.savefig(f"{OUT}/tradeoff_curves.png", dpi=150); plt.close()print("Saved tradeoff_curves.png")test_pool["proba_m"] = best_proba

### Chart 4 of 10 - caught vs. missed events per well, and trips by cause

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))w_lbls = [f"Well {r['well']}" for _, r in per_well_df.iterrows()]axes[0].bar(w_lbls, per_well_df["caught"], label="caught", color="#2A9D8F")axes[0].bar(w_lbls, per_well_df["missed"], bottom=per_well_df["caught"], label="missed", color="#C1666B")for i, r in per_well_df.reset_index().iterrows():    axes[0].text(i, r["trips"] + 0.8, f"{r['caught_pct']}%", ha="center", fontsize=9)axes[0].set_title("Trips caught vs missed, per held-out well"); axes[0].set_ylabel("trips")axes[0].legend(fontsize=8); axes[0].grid(axis="y", **GRID_KW)cause_by_well = (test_trips.groupby(["well_id", "failure_cause"]).size().unstack(fill_value=0))cause_by_well.plot(kind="bar", stacked=True, ax=axes[1], colormap="tab20", width=0.6)axes[1].set_title("Trips by cause, per held-out well")axes[1].set_xlabel(""); axes[1].set_ylabel("trips")axes[1].tick_params(axis="x", rotation=0)axes[1].legend(fontsize=7, ncol=2); axes[1].grid(axis="y", **GRID_KW)plt.tight_layout(); plt.savefig(f"{OUT}/per_well_performance.png", dpi=150); plt.close()print("Saved per_well_performance.png")

### Chart 5 of 10 - catch rate by cause, and lead-time distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))cc = cause_catch.sort_values("caught_pct")axes[0].barh(cc.index.tolist(), cc["caught_pct"], color="#4C72B0")for i, (idx, row) in enumerate(cc.iterrows()):    axes[0].text(row["caught_pct"] + 1.5, i, f"{int(row['caught'])}/{int(row['trips'])}", va="center", fontsize=8)axes[0].set_xlabel("% of trips caught"); axes[0].set_title("Catch rate by failure cause")axes[0].set_xlim(0, 105); axes[0].grid(axis="x", **GRID_KW)if lead_times:    axes[1].hist(lead_times, bins=18, color="#2A9D8F", edgecolor="white")    axes[1].axvline(float(np.mean(lead_times)), color="#C96A16", linestyle="--",                    label=f"mean {np.mean(lead_times):.0f} min")    axes[1].axvline(float(np.median(lead_times)), color="#16233A", linestyle=":",                    label=f"median {np.median(lead_times):.0f} min")    axes[1].legend(fontsize=8)axes[1].set_xlabel("Lead time before trip (min)"); axes[1].set_ylabel("caught trips")axes[1].set_title("How much warning the caught trips gave"); axes[1].grid(axis="y", **GRID_KW)plt.tight_layout(); plt.savefig(f"{OUT}/cause_and_leadtime.png", dpi=150); plt.close()print("Saved cause_and_leadtime.png")

### Chart 6 of 10 - predicted risk timeline for each test well (10-day window around a real trip)

In [ ]:
fig, axes = plt.subplots(len(TEST_WELLS), 1, figsize=(11, 2.6 * len(TEST_WELLS)), sharex=False)if len(TEST_WELLS) == 1:    axes = [axes]for ax, wid in zip(axes, TEST_WELLS):    wf = test_pool[test_pool["well_id"] == wid]    wtrips = test_trips[test_trips["well_id"] == wid]    if len(wtrips) == 0:        continue    centre = wtrips.iloc[len(wtrips) // 2]["timestamp"]    lo, hi = centre - pd.Timedelta(days=5), centre + pd.Timedelta(days=5)    seg = wf[(wf["timestamp"] >= lo) & (wf["timestamp"] <= hi)]    ax.plot(seg["timestamp"], seg["proba_m"], color="#16233A", linewidth=0.7)    ax.axhline(test_thresh_map[wid], color="#C96A16", linestyle="--", linewidth=1,               label=f"this well's threshold ({test_thresh_map[wid]:.3f})")    shown = False    for _, tr in wtrips[(wtrips["timestamp"] >= lo) & (wtrips["timestamp"] <= hi)].iterrows():        ax.axvline(tr["timestamp"], color="#C1666B", linewidth=1.2, alpha=0.85,                   label=None if shown else "actual trip")        shown = True    ax.set_title(f"Well {wid} - predicted risk over a 10-day window ({specs.loc[wid, 'dominant_causes']})", fontsize=9)    ax.set_ylabel("risk"); ax.legend(fontsize=7, loc="upper left"); ax.grid(**GRID_KW)plt.tight_layout(); plt.savefig(f"{OUT}/well_risk_timelines.png", dpi=150); plt.close()print("Saved well_risk_timelines.png")

### Chart 7 of 10 - feature importance for the winning model

In [ ]:
best_est = models[best_name]imp = Noneif hasattr(best_est, "feature_importances_"):    imp = pd.Series(best_est.feature_importances_, index=STAGE1_FEATURES)elif hasattr(best_est, "coef_"):    imp = pd.Series(np.abs(best_est.coef_[0]), index=STAGE1_FEATURES)if imp is not None:    top = imp.sort_values(ascending=False).head(18)[::-1]    fig, ax = plt.subplots(figsize=(7.6, 5.4))    ax.barh(top.index.tolist(), top.values, color="#4C72B0")    ax.set_title(f"What drives the detector - top 18 features ({best_name})", fontsize=10)    ax.tick_params(axis="y", labelsize=7.5); ax.grid(axis="x", **GRID_KW)    plt.tight_layout(); plt.savefig(f"{OUT}/feature_importance.png", dpi=150); plt.close()    print("Saved feature_importance.png")

### Chart 8 of 10 - raw sensor traces around one real trip (the physics behind an alarm)

### Bonus chart - Before & After the trip: gradual precursor vs. abrupt failureTwo real trips from the held-out test wells, same 8-hour window before the shutdown:- **Left**: a `VIBRATION` trip - one of the causes with a designed precursor leak, so vibration should visibly climb well before the trip.- **Right**: a `LOW_VOLTAGE` trip - a cause with NO precursor by design (a sudden external supply-grid event), so voltage should look flat right up until it drops.This is the visual evidence for the precursor-leak design choice from Step 3: some failure modes genuinely give an early warning, others genuinely don't - shown here from real generated data instead of just asserted in a comment.

In [ ]:
leaky_cause = "VIBRATION"
nonleaky_cause = "LOW_VOLTAGE"

def pick_trip(cause_name):
    rows = test_trips[test_trips["failure_cause"] == cause_name]
    return rows.iloc[len(rows) // 2] if len(rows) > 0 else None

trip_leaky = pick_trip(leaky_cause)
trip_nonleaky = pick_trip(nonleaky_cause)

def sensor_segment(trip, sensor_col):
    wid, st = trip["well_id"], trip["timestamp"]
    return test_pool[(test_pool["well_id"] == wid) &
                      (test_pool["timestamp"] >= st - pd.Timedelta(hours=8)) &
                      (test_pool["timestamp"] <= st + pd.Timedelta(hours=1))]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

if trip_leaky is not None:
    seg = sensor_segment(trip_leaky, "vibration_g")
    axes[0].plot(seg["timestamp"], seg["vibration_g"], color="#4C72B0", linewidth=1)
    axes[0].axvline(trip_leaky["timestamp"], color="#C1666B", linewidth=1.3)
    axes[0].set_title(f"{leaky_cause} (has a precursor leak)\nwell {trip_leaky['well_id']}")
    axes[0].set_ylabel("vibration_g")
else:
    axes[0].set_title(f"No {leaky_cause} trip found in test wells")

if trip_nonleaky is not None:
    seg = sensor_segment(trip_nonleaky, "voltage_v")
    axes[1].plot(seg["timestamp"], seg["voltage_v"], color="#4C72B0", linewidth=1)
    axes[1].axvline(trip_nonleaky["timestamp"], color="#C1666B", linewidth=1.3)
    axes[1].set_title(f"{nonleaky_cause} (no precursor by design)\nwell {trip_nonleaky['well_id']}")
    axes[1].set_ylabel("voltage_v")
else:
    axes[1].set_title(f"No {nonleaky_cause} trip found in test wells")

for ax in axes:
    ax.set_xlabel("time"); ax.tick_params(axis="x", rotation=20)

plt.suptitle("Before & After the trip: gradual precursor vs. abrupt, no-warning failure")
plt.tight_layout()
plt.savefig(f"{OUT}/before_after_precursor.png", dpi=150)
plt.close()
print("Saved before_after_precursor.png")

In [ ]:
tr0 = test_trips.iloc[len(test_trips) // 2]wid0, st0 = tr0["well_id"], tr0["timestamp"]seg = test_pool[(test_pool["well_id"] == wid0) &                (test_pool["timestamp"] >= st0 - pd.Timedelta(hours=8)) &                (test_pool["timestamp"] <= st0 + pd.Timedelta(hours=1))]trace_cols = [("motor_current_a", "Motor current (A)"), ("vibration_g", "Vibration (g)"),              ("Pd_discharge_psi", "Discharge pressure (psi)"), ("voltage_v", "Supply voltage (V)")]fig, axes = plt.subplots(len(trace_cols) + 1, 1, figsize=(10, 2.0 * (len(trace_cols) + 1)), sharex=True)for ax, (col, lbl) in zip(axes, trace_cols):    if col in seg.columns:        ax.plot(seg["timestamp"], seg[col], color="#3E5C76", linewidth=0.9)    ax.axvline(st0, color="#C1666B", linewidth=1.3)    ax.set_ylabel(lbl, fontsize=8); ax.grid(**GRID_KW)axes[-1].plot(seg["timestamp"], seg["proba_m"], color="#16233A", linewidth=1.2)axes[-1].axhline(test_thresh_map[wid0], color="#C96A16", linestyle="--", linewidth=1)axes[-1].axvline(st0, color="#C1666B", linewidth=1.3)axes[-1].set_ylabel("model risk", fontsize=8); axes[-1].set_xlabel("time"); axes[-1].grid(**GRID_KW)axes[0].set_title(f"Well {wid0} - sensors and model risk around a real {tr0['failure_cause']} trip", fontsize=10)plt.tight_layout(); plt.savefig(f"{OUT}/sensor_traces_trip.png", dpi=150); plt.close()print("Saved sensor_traces_trip.png")AVOIDABLE_DOWNTIME_HR = 3.0downtime_avoided_hr = caught_events * AVOIDABLE_DOWNTIME_HRprint(f"Estimated downtime avoided: ~{downtime_avoided_hr:.0f} hours across held-out wells")event_log_df = pd.DataFrame(event_log)

### Stage 2 - root-cause classificationTrained only on rows where the simulator's own state machine says the well is actively in `onset` for a given cause (a clean, unambiguous label), plus equipment features (gas-separator efficiency, pump stage count, horsepower) as prior knowledge to help tell apart causes with similar symptoms (like `GAS_LOCK` and `UNDERLOAD`, which both show up as a current drop).

In [ ]:
print("\n=== SUPERVISED STAGE 2: Root-cause classification ===")cause_fit["onset_cause"] = cause_fit["onset_cause"].astype(str)cause_test["onset_cause"] = cause_test["onset_cause"].astype(str)# ---- equipment configuration as prior knowledge ----# GAS_LOCK and UNDERLOAD both present as a motor-current dip and are close to# indistinguishable from the sensor trace alone - which is why both sat at# 0.00 recall. But they are not equally likely on every well: a well with a# weak or absent gas separator is far more gas-lock-prone than one with an# 85%-efficient rotary separator. That is real prior knowledge an engineer# uses when interpreting the same symptom, and the model had no access to it.# These are per-well constants available for any new well (they come off the# equipment spec sheet), so they transfer to unseen wells by construction.EQUIP_FEATURES = ["gas_separator_efficiency_pct", "pump_n_stages", "motor_rated_hp"]for frame in (cause_fit, cause_test):    for col in EQUIP_FEATURES:        frame[col] = frame["well_id"].map(specs[col].to_dict()).astype("float32")STAGE2_FEATURES = ALL_FEATURES + EQUIP_FEATURESprint(f"Stage 2 features: {len(ALL_FEATURES)} sensor/rolling + {len(EQUIP_FEATURES)} equipment-spec")print("Training rows per cause (fit set, true onset rows only):")print(cause_fit["onset_cause"].value_counts())

### Train a Random Forest cause classifier, and evaluate it on the fully held-out test wells

In [ ]:
le = LabelEncoder()y_cause_fit = le.fit_transform(cause_fit["onset_cause"])X_cause_fit = cause_fit[STAGE2_FEATURES].to_numpy(dtype="float32")cause_model = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced_subsample", random_state=7, n_jobs=2)cause_model.fit(X_cause_fit, y_cause_fit)X_cause_test = cause_test[STAGE2_FEATURES].to_numpy(dtype="float32")y_cause_test_true = cause_test["onset_cause"]known_labels = set(le.classes_)valid_mask = y_cause_test_true.isin(known_labels)X_cause_test_valid = X_cause_test[valid_mask.values]y_cause_test_valid = le.transform(y_cause_test_true[valid_mask])cause_pred = cause_model.predict(X_cause_test_valid)cause_acc = accuracy_score(y_cause_test_valid, cause_pred)print(f"\nRoot-cause accuracy on held-out wells (n={len(y_cause_test_valid)}): {cause_acc:.3f}")# some causes (e.g. HIGH_TEMP) never occur in held-out wells by design - pass# the full label set explicitly so classes with 0 test support still print# a (0-support) row instead of crashing on a class-count mismatchcause_report = classification_report(    y_cause_test_valid, cause_pred, labels=range(len(le.classes_)),    target_names=le.classes_, zero_division=0,)print(cause_report)

### Save every result and summary into one `results.txt` file

In [ ]:
FREQ_RELATED = {"GAS_LOCK": True, "UNDERLOAD": False, "HIGH_TEMP": True,                 "HIGH_DISCHARGE": True, "VIBRATION": False, "LOW_VOLTAGE": False}# ============================================================# Save results# ============================================================with open(f"{OUT}/results.txt", "w") as f:    f.write(f"Horizon H = {H} minutes\n")    f.write(f"Train wells (fully seen): {TRAIN_WELLS}\n")    f.write(f"Held-out test wells (never seen): {TEST_WELLS}\n")    f.write("Time-holdout within train wells: final 6 weeks used as validation\n")    f.write("Features: z-score normalized PER WELL (fit-period baseline for train wells; "             "first-30-days calibration baseline for held-out test wells)\n\n")    f.write("=== Well equipment specs & safe operating band ===\n")    for well_id in specs.index:        s = specs.loc[well_id]        f.write(f"  Well {well_id}: Motor={s['motor_model']}, Protector={s['protector_model']}, "                 f"GasSeparator={s['gas_separator_model']} ({s['gas_separator_efficiency_pct']}% eff), "                 f"Pump={s['pump_model']}\n")        f.write(f"    Safe Hz band: [{s['safe_min_hz']:.0f}, {s['safe_max_hz']:.0f}] Hz, "                 f"BEP={s['pump_bep_hz']:.0f} Hz, dominant causes: {s['dominant_causes']}\n")    f.write("\n=== UNSUPERVISED: Clustering ===\n")    f.write(f"K-Means (k=6) silhouette (normalized features): {km_sil:.3f}\n")    f.write("K-Means cluster -> failure-within-H rate:\n")    f.write(cluster_failure_rate.to_string() + "\n")    f.write(f"DBSCAN: {n_db_clusters} clusters found. Failure rate: "             f"outliers={outlier_failure_rate.get(True, float('nan')):.4f} vs. "             f"clustered-normal={outlier_failure_rate.get(False, float('nan')):.4f}\n\n")    f.write("=== SUPERVISED STAGE 1: Failure detection - models vs. baselines ===\n")    f.write(results_df.to_string(index=False) + "\n\n")    f.write(f"Best model (selected on VALIDATION ROC-AUC, never test): {best_name}\n")    f.write(f"Stage 1 features: {len(STAGE1_FEATURES)} ({'relative-only' if USE_RELATIVE_FEATURES else 'raw + rolling'})\n")    f.write(f"Alarm policy (tuned on validation, cost-weighted {MISS_COST_RATIO:.0f}:1): "             f"per-well top {100*(1-Q):.1f}% of that well's own scores, "             f"persistence={N_PERSIST} consecutive minutes\n")    f.write("Per-well alert thresholds (held-out): "             + ", ".join(f"well {w}: {t:.4f}" for w, t in sorted(test_thresh_map.items())) + "\n")    f.write(f"Minute-level confusion matrix (pre-persistence): TN={tn} FP={fp} FN={fn} TP={tp}\n")    f.write(f"Trips in held-out wells: {n_events}\n")    f.write(f"Caught (>=1 alarm in prior {ALERT_WINDOW_MIN} min): {caught_events} ({100*caught_events/max(n_events,1):.0f}%)\n")    f.write(f"Missed failures: {missed_events} ({100*missed_events/max(n_events,1):.0f}%)\n")    f.write(f"False-alarm EVENTS: {false_alarm_events} (~{fa_per_well_month:.1f} per well per month)\n")    f.write(f"Total alarms raised: {final['n_alarms']}\n")    if lead_times:        f.write(f"Average lead time: {np.mean(lead_times):.1f} min (median {np.median(lead_times):.1f} min)\n")    f.write(f"Estimated downtime avoided: ~{downtime_avoided_hr:.0f} hours across held-out wells\n\n")    f.write("Alarm-policy grid searched on validation (top 10 by cost):\n")    f.write(policy_df.head(10).to_string(index=False) + "\n\n")    f.write("Trade-off curve on held-out wells (REPORTED ONLY - policy above was chosen on validation):\n")    f.write(tradeoff_df.to_string(index=False) + "\n\n")    f.write("=== PER-MODEL event-level performance on held-out wells (each at its own validation-tuned policy) ===\n")    f.write(per_model_df.to_string(index=False) + "\n\n")    f.write("=== PER-WELL breakdown (selected model, shipped policy) ===\n")    f.write(per_well_df.to_string(index=False) + "\n\n")    f.write("=== Catch rate by failure cause (selected model) ===\n")    f.write(cause_catch.to_string() + "\n\n")    f.write("=== SUPERVISED STAGE 2: Root-cause classification ===\n")    f.write(f"Accuracy on held-out wells: {cause_acc:.3f}\n")    f.write(cause_report + "\n")    f.write("=== Frequency-related vs. not (per cause) - drives the recommendation branch ===\n")    for c, is_freq in FREQ_RELATED.items():        f.write(f"  {c}: {'frequency adjustment applicable (clip to well safe_min_hz/safe_max_hz)' if is_freq else 'NOT frequency-related -> flag inspection instead'}\n")print("\nSaved results.txt")

### Chart 9 of 10 - timeline for one representative test well: actual sensors vs. predicted risk

In [ ]:
rep_well = TEST_WELLS[0]rep_events = event_log_df[(event_log_df["well_id"] == rep_well) & (event_log_df["caught"] == True)]if len(rep_events) > 0:    target_st = rep_events.iloc[len(rep_events) // 2]["timestamp"]    win_start, win_end = target_st - pd.Timedelta(hours=6), target_st + pd.Timedelta(hours=3)    seg = test_pool[(test_pool["well_id"] == rep_well) & (test_pool["timestamp"] >= win_start) & (test_pool["timestamp"] <= win_end)]    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)    axes[0].plot(seg["timestamp"], seg["vibration_g"], color="#3E5C76", linewidth=1, label="vibration (raw)")    axes[0].axvline(target_st, color="#C96A16", linestyle="--", linewidth=1.5, label="Actual trip")    axes[0].legend(fontsize=8, loc="upper left")    axes[0].set_title(f"Held-out well {rep_well} — actual vs. predicted risk ({best_name}, relative features)")    axes[1].plot(seg["timestamp"], seg["proba"], color="#16233A", linewidth=1.5, label="predicted trip risk")    axes[1].axhline(test_thresh_map[rep_well], color="gray", linestyle=":", linewidth=1, label="Alert threshold (this well)")    axes[1].axvline(target_st, color="#C96A16", linestyle="--", linewidth=1.5)    axes[1].set_ylabel("Predicted\ntrip probability"); axes[1].set_xlabel("Time")    axes[1].legend(fontsize=8, loc="upper left")    plt.tight_layout()    plt.savefig(f"{OUT}/heldout_well_timeline.png", dpi=150)    plt.close()    print(f"Saved timeline chart for held-out well {rep_well}")

### Chart 10 of 10 - confusion matrix for the Stage 2 root-cause classifier

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))cm_cause = confusion_matrix(y_cause_test_valid, cause_pred, labels=range(len(le.classes_)))im = ax.imshow(cm_cause, cmap="Blues")ax.set_xticks(range(len(le.classes_))); ax.set_xticklabels(le.classes_, rotation=45, ha="right", fontsize=8)ax.set_yticks(range(len(le.classes_))); ax.set_yticklabels(le.classes_, fontsize=8)ax.set_xlabel("Predicted cause"); ax.set_ylabel("Actual cause")ax.set_title("Stage 2: Root-cause classification (held-out wells, normalized)")for i in range(cm_cause.shape[0]):    for j in range(cm_cause.shape[1]):        ax.text(j, i, cm_cause[i, j], ha="center", va="center",                 color="white" if cm_cause[i, j] > cm_cause.max() / 2 else "black", fontsize=8)plt.tight_layout()plt.savefig(f"{OUT}/cause_confusion_matrix.png", dpi=150)plt.close()print("Saved cause_confusion_matrix.png")

## Step 5 — Results and charts

### Print the full contents of `results.txt`

In [ ]:
print(open('./esp_multiwell/results.txt').read())

### Show each chart in its own cellInstead of one loop that displays all 10 images, each image below gets its own markdown title cell and its own code cell.

**Each model at its own tuned alarm policy**  `model_comparison_events.png`

**Before & After: per-well normalization**  `before_after_normalization.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/before_after_normalization.png'))

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/model_comparison_events.png'))

**Operating-point trade-off curves**  `tradeoff_curves.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/tradeoff_curves.png'))

**ROC and Precision-Recall on held-out wells**  `roc_pr_curves.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/roc_pr_curves.png'))

**Predicted risk per held-out well**  `well_risk_timelines.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/well_risk_timelines.png'))

**Per-well caught vs missed, and trips by cause**  `per_well_performance.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/per_well_performance.png'))

**Catch rate by cause, and lead-time distribution**  `cause_and_leadtime.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/cause_and_leadtime.png'))

**Sensors and model risk around a real trip**  `sensor_traces_trip.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/sensor_traces_trip.png'))

**What the detector keys on**  `feature_importance.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/feature_importance.png'))

**Unsupervised: K-Means clusters (PCA)**  `clustering_pca_normalized.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/clustering_pca_normalized.png'))

**Stage 2: root-cause confusion matrix**  `cause_confusion_matrix.png`

In [ ]:
from IPython.display import Image, display
display(Image('./esp_multiwell/cause_confusion_matrix.png'))

## Notes for whoever picks this up**Where things stand, honestly.** Stage 1 catches roughly a third of trips at ~60 nuisance alarms per well per month, with ~110 minutes of average warning. That is a real result, not a solved problem — and the trade-off curve printed in Step 5 shows what other operating points buy you.**Read the per-cause catch rates.** VIBRATION 55%, GAS_LOCK 38%, HIGH_DISCHARGE 36%, UNDERLOAD 28%, LOW_VOLTAGE 19%. That ordering is the best evidence the pipeline learns real physics: causes with gradual build-up are predictable, the sudden electrical one isn't. The 19% is not a bug to tune away.**Model choice is not obvious.** Logistic Regression catches the most trips with the fewest alarms, but its *median lead time is 6 minutes* — it notices failures rather than predicting them. Random Forest gives 40% caught with ~163 minutes of median warning. Which one is "best" depends entirely on whether the warning time is usable.**Stage 2 is half-solved.** VIBRATION is classified almost perfectly (99% precision / 96% recall on unseen wells). GAS_LOCK, HIGH_DISCHARGE and UNDERLOAD still collapse into LOW_VOLTAGE. GAS_LOCK and UNDERLOAD may be close to *fundamentally* confusable here — both present as a motor-current dip.**Things already tried and rejected** (don't redo them without a new angle):- *Frequency-detrended residual features* — Stage 2 barely moved, Stage 1 got much worse.- *Relative-only features* (no absolute sensor levels) — closed the val→test gap from the wrong end: validation fell, test didn't rise, catch rate dropped to 14%. Still in the code behind `USE_RELATIVE_FEATURES = False`.- *F1-maximised alert threshold* — degenerates to near-zero recall when positives are under 1% of rows.**Most promising next steps:** cause-specific engineered features for the three collapsed causes; selecting the model on event-level cost rather than minute-level ROC-AUC; and a survival-analysis formulation instead of a fixed-horizon binary label.**To add a well:** edit the config in Step 1, then re-run Steps 2 → 4. Step 3 only simulates the new well. To use real historical data instead, match the schema (`well_id, split, timestamp, state, frequency_hz, motor_current_a, voltage_v, Ti_intake_temp_f, Tm_motor_temp_f, Pi_intake_psi, Pd_discharge_psi, vibration_g, shutdown_event, failure_cause, onset_cause, safe_min_hz, safe_max_hz`) and append it to the CSV.